# Part 3 — Feature engineering and feature evaluation for user–product similarity

This notebook builds the features described in the proposal *JDsearch Feature Engineering Proposal: User–Product Similarity*. It then evaluates each feature, using only the labels already in the data, before any model is trained. It does not train the ranker. The last cell shows how the saved features would be used to train a LightGBM LambdaRank model.

**What the notebook produces.** For every one of the 15,348,049 candidate rows of the 171,175 eligible sessions, it produces 90 features in eight families (A to H) plus user context. The features are saved to Google Drive as Parquet files, one per chunk of 2,000 sessions, together with a feature dictionary and the results of the feature evaluation.

**It is standalone.** Everything is rebuilt from the two raw files, `user_behavior_data.txt` and `product_meta_data.txt`, with the same definitions as the earlier notebooks:

- the eligible sessions and the 80/10/10 split are reproduced with exactly the method of Siqian's preprocessing notebook;
- the cart-cleaning rule, the elapsed-day convention, the exact-product features and the session recency features use exactly the definitions of Part 1.

Rebuilding is preferable to reading the earlier outputs, for two reasons. First, the features need every user's history parsed action by action, and no earlier output stores the history in that form, so the raw file has to be read in any case. Second, it removes any dependence on intermediate files whose provenance may differ (for example, the candidate chunks dated 27 October). Where the earlier outputs exist (`user_split.csv`, `session_recency_features.parquet`, `candidate_recency_chunks/`), they are used only to *check* that this notebook reproduces them (Step 9).

**How memory is kept low.** A standard Colab runtime has about 12.7 GB of RAM, and a naive approach (parsing every history into Python lists) needs well over that. The notebook therefore follows five rules:

1. The raw user file is read 10,000 rows at a time, and its parsed contents are written straight to compact Parquet files on the local disk, in chunks of 2,000 sessions (the same grouping as Part 1). No step holds all sessions' histories in memory.
2. Identifiers are stored as integers (product IDs as 64-bit, session IDs as 32-bit, labels and action types as 8-bit) rather than as Python strings, which cost about 57 bytes each.
3. The large product-level tables (product attributes, the title TF-IDF matrix, item and term embeddings) are written to disk as NumPy arrays and opened *memory-mapped*. Only the rows needed for the current chunk are read into RAM.
4. Features are computed chunk by chunk and written to Drive immediately. A chunk that already exists is skipped, so an interrupted run can resume.
5. Large temporary objects are deleted as soon as each step finishes.

**Order of the steps.**

| Step | What it does | Main output |
|---|---|---|
| 0 | Setup and configuration | — |
| 1 | Stream the user file into session chunks (candidates, cleaned histories) | `stage/cand_*.parquet`, `stage/hist_*.parquet`, `sessions.parquet` |
| 2 | Reproduce the train / validation / test split | `split` column in `sessions.parquet` |
| 3 | Product tables: attributes and title TF-IDF | memory-mapped arrays |
| 4 | Embeddings: item2vec on histories, term2vec on titles | memory-mapped arrays |
| 5 | Training-set priors (family G), out-of-fold | `stage/priors_*.parquet` |
| 6 | Feature computation by chunk (families A to H, context) | `feature_chunks/feat_*.parquet` on Drive |
| 7 | Feature dictionary | `feature_dictionary_part3.csv` |
| 8 | Evaluate features (before training) | `feature_screening.csv`, `model_features.json` |
| 9 | Consistency checks against the earlier notebooks' outputs | printed comparison |
| 10 | How to train LightGBM with these outputs (example, not run) | — |

## Features that are *not* built: those that use NMF or NCF results

The NMF and NCF models of Part 2 (and of the Group1JDSearch_NJ notebook) are collaborative-filtering models. They are trained only on the matrix of (session, product, action weight) taken from `history_wid_list` and `history_type_list`. The proposal originally considered using their outputs as ranker features. As instructed, **every feature that uses NMF or NCF results is dropped**, so that the ranker's performance can be attributed to the features built here. The dropped features are:

| Dropped feature | Where it came from | Why it is dropped |
|---|---|---|
| `nmf_score` | Part 2: dot product of the NMF user and item factors | Uses NMF results |
| `ncf_score` | Part 2: sigmoid output of the NCF network | Uses NCF results |
| `hybrid_score` | Part 2: `nmf_score` + 10⁶, with `pop_score` as fallback | Uses NMF results (and the leaking `pop_score`) |
| `ncf_hybrid_score` | Part 2: `ncf_score` + 10⁶, with `pop_score` as fallback | Uses NCF results (and the leaking `pop_score`) |
| `tt_score` (two-tower) and stacked NCF scores | First draft of the proposal, family D | Neural collaborative-filtering scores of the same kind as NCF |

In addition, Part 2's full-data `pop_score` is not used. It does not come from NMF or NCF, but it counts each evaluated row's own label, which is leakage. Family G replaces it with training-set rates computed out-of-fold.

Family D (embedding similarity) is retained. Its item2vec and term2vec vectors are trained here, from histories and product titles. They make no use of the NMF or NCF models or their outputs.

## Step 0 — Setup and configuration

This step mounts Drive, installs `gensim` (used for item2vec and term2vec), and defines all paths and settings in one place.

`gensim==4.4.0` is pinned because it supports NumPy 2, which Colab now uses. Earlier versions require NumPy 1 and would downgrade Colab's NumPy.

There are two working directories:
- `P3_DIR` on Drive holds the **final outputs** (feature chunks, dictionary, evaluation results), which must survive the session.
- `LOCAL_DIR` on the Colab machine's own disk holds the **intermediate files** (session chunks, product arrays, embeddings). Local disk is much faster than Drive for repeated and memory-mapped reads, but it is erased when the runtime is recycled. If `PERSIST_INTERMEDIATES` is `True`, the intermediates are also copied to Drive after each step and restored automatically after a restart.

In [2]:
%pip install -q gensim==4.4.0

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import gc, json, math, os, shutil, time
from collections import Counter
from itertools import chain
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.preprocessing import normalize

try:
    import psutil
    def ram(label=""):
        print(f"[RAM] {label}: {psutil.Process().memory_info().rss / 1e9:.2f} GB in use")
except ImportError:
    def ram(label=""):
        pass

# ---------------- paths ----------------
PROJECT_DIR = Path("/content/drive/MyDrive/BT4222 Final/JDSearch")
RAW_DIR = PROJECT_DIR / "Raw Data"
USER_TXT = RAW_DIR / "user_behavior_data.txt"
PRODUCT_TXT = RAW_DIR / "product_meta_data.txt"
PRODUCT_CACHE_DIR = RAW_DIR / "product_chunks"      # optional Parquet copy of the product file (Part 1)
OUTPUT_DIR = PROJECT_DIR / "processed"              # outputs of the earlier notebooks (used only for checks)

P3_DIR = OUTPUT_DIR / "part3_features"              # final outputs of this notebook (Drive)
FEAT_DIR = P3_DIR / "feature_chunks"
LOCAL_DIR = Path("/content/part3_local")            # intermediates (local disk)
STAGE_DIR = LOCAL_DIR / "stage"
ARR_DIR = LOCAL_DIR / "arrays"
TITLE_DIR = LOCAL_DIR / "titles"
BACKUP_DIR = P3_DIR / "intermediates_backup"
for d in [P3_DIR, FEAT_DIR, STAGE_DIR, ARR_DIR, TITLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

PERSIST_INTERMEDIATES = True   # copy intermediates to Drive after each step (costs Drive space, saves rebuild time)

# ---------------- settings (see the proposal, Sections 3-4) ----------------
SEED = 42                      # same seed as Siqian's split
SESSIONS_PER_CHUNK = 2000      # same grouping as Part 1 -> 86 chunks
READ_ROWS = 10_000             # rows of the raw user file read at a time
CART_MAX_DAYS = 90             # Part 1: cart actions older than this are anomalies and removed
TYPE_CODE = {"CLICK": 0, "CART": 1, "ORD": 2, "FLW": 3}
TYPE_NAMES = ["click", "cart", "ord", "flw"]
TYPE_WEIGHT = np.array([1.0, 3.0, 5.0, 2.0])   # CLICK 1, CART 3, ORD 5, FLW 2; unknown types get 1
TAU_DAYS = 30.0                # decay constant of the action weight
TERM_SEP = "\x18"              # separator of terms in titles and queries
TITLE_MIN_DF = 5               # title terms in fewer products than this are dropped
EMB_DIM, EMB_WINDOW, EMB_EPOCHS = 64, 5, 5
I2V_MIN_COUNT = 5              # products seen fewer times in all histories get no item2vec vector
PRIOR_ALPHA = 20.0             # smoothing of the training-set rates
N_FOLDS = 5                    # folds for the out-of-fold training-set rates
BURST_GAP_DAYS = 30 / 1440     # 30 minutes
SCREEN_SESSIONS = 10_000       # training sessions sampled for the feature evaluation
EXPECTED = {"sessions": 173_831, "eligible": 171_175, "candidate_rows": 15_348_049, "carts_removed": 1_177}


def backup(*names):
    # Copy intermediate folders from local disk to Drive.
    if PERSIST_INTERMEDIATES:
        for n in names:
            src = LOCAL_DIR / n
            if src.is_dir():
                shutil.copytree(src, BACKUP_DIR / n, dirs_exist_ok=True)
            elif src.exists():
                (BACKUP_DIR).mkdir(parents=True, exist_ok=True)
                shutil.copy2(src, BACKUP_DIR / n)


def restore(*names):
    # Bring intermediates back from Drive after a runtime restart; returns True if all are available.
    ok = True
    for n in names:
        dst, src = LOCAL_DIR / n, BACKUP_DIR / n
        if not dst.exists() and src.exists():
            (shutil.copytree(src, dst) if src.is_dir() else shutil.copy2(src, dst))
        ok = ok and dst.exists()
    return ok


def done(flag):
    return (LOCAL_DIR / f"{flag}.done").exists() or restore(f"{flag}.done")


def mark_done(flag):
    (LOCAL_DIR / f"{flag}.done").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))


ram("setup")

Mounted at /content/drive
[RAM] setup: 0.25 GB in use


### The feature list

The 90 features, grouped by family as in Section 4 of the proposal, are defined once here. Every later step refers to this list, so the saved files, the dictionary and the evaluation always agree. The identifying columns `session_id`, `wid`, `label` and `split` are stored alongside the features but are never features themselves.

In [4]:
ATTRS = ["brand", "shop", "cate1", "cate2", "cate3", "cate4"]       # order of the attribute columns
ATTR_SOURCE = ["brand_id", "shop_id", "cate_id_1", "cate_id_2", "cate_id_3", "cate_id_4"]
BRAND, CATE3 = 0, 4                                                  # column positions used by E, F, G, H
H_BASE = ["brand_aff", "shop_aff", "cate3_aff", "title_cos_profile", "title_cos_max", "i2v_cos_max", "item_ctr_train"]

FAMILIES = {
    "A": ["exact_wid_click_0_7d", "exact_wid_cart_0_7d", "exact_wid_ord_0_7d", "exact_wid_cart_7_90d",
          "exact_wid_ord_7_90d", "exact_wid_ord_90_365d", "exact_wid_last_seen_days"],
    "B": [f"{k}_{s}" for k in ATTRS for s in ["aff", "share", "last_seen_days", "ord_cnt"]],
    "C": ["title_cos_profile", "title_cos_profile_ord", "title_cos_max", "title_hist_qry_overlap"],
    "D": ["i2v_cos_mean", "i2v_cos_max", "i2v_cos_last5", "t2v_cos_profile"],
    "E": ["last_same_brand", "last_same_cate3", "last_title_cos", "last_i2v_cos"],
    "F": ["qhist_n_actions", "qhist_wid_cnt", "qhist_brand_aff", "qhist_cate3_aff"],
    "G": ["item_ctr_train", "brand_ctr_train", "cate3_ctr_train", "item_impr_train"],
    "H": [f"{f}_slate_{s}" for f in H_BASE for s in ["pct", "z"]] + ["cate3_slate_share", "slate_size"],
    "context": [f"{t}_count_{w}" for w in ["0_7d", "7_90d", "90_365d"] for t in TYPE_NAMES]
               + ["history_len_clean", "history_span_days", "days_since_last_interaction",
                  "share_click", "share_cart", "share_ord", "share_flw",
                  "cate3_entropy", "n_distinct_brands", "qryless_share", "burst_len"],
}
FEATURE_COLUMNS = [f for fam in FAMILIES.values() for f in fam]
FAMILY_OF = {f: fam for fam, cols in FAMILIES.items() for f in cols}
ID_COLUMNS = ["session_id", "wid", "label", "split"]
assert len(FEATURE_COLUMNS) == len(set(FEATURE_COLUMNS)) == 90
print({k: len(v) for k, v in FAMILIES.items()}, "total", len(FEATURE_COLUMNS))

{'A': 7, 'B': 24, 'C': 4, 'D': 4, 'E': 4, 'F': 4, 'G': 4, 'H': 16, 'context': 23} total 90


## Step 1 — Stream the user file into session chunks

**What this step does.** It reads `user_behavior_data.txt` 10,000 rows at a time and, for every session, does the following:

1. **Session identifier.** The row number of the session in the file becomes `session_id`, as in Siqian's notebook.
2. **Candidates.** It splits `candidate_wid_list` and `candidate_label_list` on `_` into one row per candidate. Product IDs are converted to integers and labels to 0–3.
3. **Eligibility.** A session is kept only if it has at least one positive (label above 0) and at least one negative (label 0) candidate, as in Siqian's notebook. All-negative and all-positive sessions carry no ranking information.
4. **Candidate order.** The candidates of each session are shuffled into a random order, as Part 1 does, because the display order reveals the label (the first candidate is positive in about 99% of sessions). The original display position is kept in a separate column, `position`, which is used *only* by the leakage check in Step 8 and is never a feature.
5. **History.** It splits the four history columns into one row per past action and computes the **elapsed days** of each action with Part 1's convention. `history_time_list` has one more entry than there are actions. The first entry is not used, entry *i*+1 is the gap after action *i*, and the last entry is the gap between the final action and the query. The elapsed days of action *i* are therefore the sum of all gaps from position *i*+1 to the end, divided by 86,400. When the four lists differ in length, they are truncated to the shortest, as Part 1's `zip` does. A missing past query counts as `-1`.
6. **Cleaning.** It removes cart actions recorded more than 90 days before the query (Part 1's rule; 1,177 actions on the full data).

Eligible sessions are collected in their original order and written in groups of 2,000. This is exactly Part 1's batching, giving 86 chunks. Each chunk is written as two Parquet files on local disk: `cand_XXX.parquet` (session_id, wid, label, position) and `hist_XXX.parquet` (session_id, wid, type, days, hqry). The session-level data (session_id, chunk, test query, slate size) go to `sessions.parquet`. The set of all product IDs that occur is also saved, so that Step 3 loads metadata only for those products.

**Two checks on the data are printed.** The first is the share of final time gaps that are negative, which should be 0 if the elapsed-day convention is right. The second is the share of CLICK actions that fall within 7 days of the query, which should be close to 100% given how the log was recorded.

**Memory.** At most 10,000 raw rows, plus the pending part of one chunk, are held in memory at any time.

In [5]:
def split_flat(series, sep):
    # Split each string of a Series on `sep`; return the flat token array and the number of tokens per row.
    vals = series.fillna("").to_numpy(dtype=object)
    lists = [v.split(sep) if v else [] for v in vals]
    lens = np.fromiter((len(l) for l in lists), dtype=np.int64, count=len(lists))
    flat = np.array(list(chain.from_iterable(lists)), dtype=object)
    return flat, lens


def within_pos(lens):
    # Position of each token within its own row (0, 1, 2, ... restarting for every row).
    starts = np.cumsum(lens) - lens
    return np.arange(lens.sum(), dtype=np.int64) - np.repeat(starts, lens)


def to_int64(tokens, what):
    # Convert ID tokens to int64; stop with a clear message if any ID is not an integer.
    s = pd.to_numeric(pd.Series(tokens, dtype=object), errors="coerce")
    if s.isna().any():
        bad = pd.Series(tokens)[s.isna()].head(3).tolist()
        raise ValueError(f"Non-integer {what} found, e.g. {bad}. The notebook stores IDs as integers to save memory.")
    return s.to_numpy(dtype=np.int64)


def select_first(flat, lens, keep_n):
    # Keep the first keep_n[i] tokens of row i (the truncation that Part 1's zip performs).
    pos = within_pos(lens)
    return flat[pos < np.repeat(keep_n, lens)]


def parse_block(df, sid0):
    # Parse one block of raw rows. Returns candidate rows, cleaned history rows and session rows
    # for the eligible sessions only, plus counters for the printed checks.
    n = len(df)
    local = np.arange(n)
    stats = Counter()

    # ---- candidates and eligibility (Siqian) ----
    cw, cn = split_flat(df["candidate_wid_list"], "_")
    cl, cln = split_flat(df["candidate_label_list"], "_")
    lab = pd.to_numeric(pd.Series(cl, dtype=object)).to_numpy()
    l_local = np.repeat(local, cln)
    n_pos = np.bincount(l_local, weights=(lab > 0), minlength=n)
    n_neg = np.bincount(l_local, weights=(lab == 0), minlength=n)
    eligible = (n_pos > 0) & (n_neg > 0)
    # as in Siqian's notebook, IDs and labels must be aligned in every eligible session
    assert (cn == cln)[eligible].all(), "candidate IDs and labels are misaligned in an eligible session"
    c_local = np.repeat(local, cn)
    lab = lab[eligible[l_local]]
    stats["sessions"] += n
    stats["eligible"] += int(eligible.sum())
    stats["all_negative"] += int((n_pos == 0).sum())
    stats["all_positive"] += int((n_neg == 0).sum())

    keep_c = eligible[c_local]
    cand = pd.DataFrame({
        "session_id": (sid0 + c_local[keep_c]).astype(np.int32),
        "wid": to_int64(cw[keep_c], "candidate product ID"),
        "label": lab.astype(np.int8),
        "position": within_pos(cn)[keep_c].astype(np.int16),
    })

    # ---- history: parse, align, elapsed days (Part 1 convention) ----
    hw, hwn = split_flat(df["history_wid_list"], "_")
    ht, htn = split_flat(df["history_type_list"], "_")
    hq, hqn = split_flat(df["history_qry_list"], "_")
    tm, tmn = split_flat(df["history_time_list"], "_")
    t_all = to_int64(tm, "time gap")
    t_pos = within_pos(tmn)
    gaps = t_all[t_pos >= 1]                          # entries 1..n of each row
    g_len = np.maximum(tmn - 1, 0)
    rev = np.cumsum(gaps[::-1])[::-1]                 # sum of this and all later gaps (across rows)
    after = np.append(rev, 0)[np.cumsum(g_len)]       # part of that sum belonging to later rows
    days_all = (rev - np.repeat(after, g_len)) / 86400.0
    has_t = tmn > 0
    last_gap = t_all[(np.cumsum(tmn) - 1)[has_t]]
    stats["rows_with_times"] += int(eligible[has_t].sum())
    stats["negative_last_gap"] += int((last_gap < 0)[eligible[has_t]].sum())

    n_act = np.minimum(np.minimum(hwn, htn), g_len)   # Part 1 zips wid, type and days
    h_wid = to_int64(select_first(hw, hwn, n_act), "history product ID")
    h_type = pd.Series(select_first(ht, htn, n_act)).map(TYPE_CODE).fillna(-1).to_numpy(np.int8)
    h_days = select_first(days_all, g_len, n_act)
    # past queries: pad with '-1' where the query list is shorter
    out_start = np.cumsum(n_act) - n_act
    h_qry = np.full(int(n_act.sum()), "-1", dtype=object)
    q_pos = within_pos(hqn)
    q_ok = q_pos < np.repeat(n_act, hqn)
    h_qry[np.repeat(out_start, hqn)[q_ok] + q_pos[q_ok]] = hq[q_ok]
    h_local = np.repeat(local, n_act)

    # cleaning (Part 1): remove carts older than 90 days; then keep eligible sessions
    old_cart = (h_type == TYPE_CODE["CART"]) & (h_days > CART_MAX_DAYS)
    keep_h = eligible[h_local] & ~old_cart
    stats["carts_removed"] += int((old_cart & eligible[h_local]).sum())
    is_click = h_type[keep_h] == TYPE_CODE["CLICK"]
    stats["clicks"] += int(is_click.sum())
    stats["clicks_within_7d"] += int((h_days[keep_h][is_click] < 7).sum())
    hist = pd.DataFrame({
        "session_id": (sid0 + h_local[keep_h]).astype(np.int32),
        "wid": h_wid[keep_h],
        "type": h_type[keep_h],
        "days": h_days[keep_h].astype(np.float32),
        "hqry": h_qry[keep_h],
    })
    sess = pd.DataFrame({
        "session_id": (sid0 + local[eligible]).astype(np.int32),
        "query": df["query"].fillna("").to_numpy(dtype=object)[eligible],
        "slate_size": cn[eligible].astype(np.int32),
    })
    return cand, hist, sess, stats

In [6]:
def write_chunk(chunk_id, cand, hist, rng):
    # Shuffle candidates within each session (display order is not kept as row order) and write both files.
    order = np.lexsort((rng.random(len(cand)), cand["session_id"].to_numpy()))
    cand = cand.iloc[order].reset_index(drop=True)
    cand.to_parquet(STAGE_DIR / f"cand_{chunk_id:03d}.parquet", index=False)
    hist.to_parquet(STAGE_DIR / f"hist_{chunk_id:03d}.parquet", index=False)
    return np.unique(cand["wid"].to_numpy()), np.unique(hist["wid"].to_numpy())


if done("step1") and restore("stage", "sessions.parquet", "arrays/needed_wids.npy", "step1_stats.json"):
    print("Step 1 already completed - skipping.")
else:
    rng = np.random.default_rng(SEED)
    totals = Counter()
    pend_c, pend_h, pend_s = [], [], []
    sess_rows, needed = [], []
    chunk_id, sid0 = 0, 0
    reader = pd.read_csv(USER_TXT, sep="\t", dtype=str, chunksize=READ_ROWS)
    for block in reader:
        cand, hist, sess, st = parse_block(block, sid0)
        sid0 += len(block)
        totals.update(st)
        pend_c.append(cand); pend_h.append(hist); pend_s.append(sess)
        n_pending = sum(len(s) for s in pend_s)
        while n_pending >= SESSIONS_PER_CHUNK:
            s_all = pd.concat(pend_s, ignore_index=True)
            c_all = pd.concat(pend_c, ignore_index=True)
            h_all = pd.concat(pend_h, ignore_index=True)
            cut = s_all["session_id"].iloc[SESSIONS_PER_CHUNK - 1]
            s_out = s_all[s_all["session_id"] <= cut].assign(chunk=chunk_id)
            uc, uh = write_chunk(chunk_id, c_all[c_all["session_id"] <= cut], h_all[h_all["session_id"] <= cut], rng)
            sess_rows.append(s_out); needed += [uc, uh]
            pend_s = [s_all[s_all["session_id"] > cut]]
            pend_c = [c_all[c_all["session_id"] > cut]]
            pend_h = [h_all[h_all["session_id"] > cut]]
            chunk_id += 1
            n_pending = len(pend_s[0])
        print(f"read {sid0:,} rows -> {chunk_id} chunks written", end="\r")
        del block, cand, hist, sess
    # final partial chunk
    s_all = pd.concat(pend_s, ignore_index=True)
    if len(s_all):
        uc, uh = write_chunk(chunk_id, pd.concat(pend_c, ignore_index=True), pd.concat(pend_h, ignore_index=True), rng)
        sess_rows.append(s_all.assign(chunk=chunk_id)); needed += [uc, uh]
        chunk_id += 1
    sessions = pd.concat(sess_rows, ignore_index=True)
    sessions.to_parquet(LOCAL_DIR / "sessions.parquet", index=False)
    np.save(ARR_DIR / "needed_wids.npy", np.unique(np.concatenate(needed)))
    totals["chunks"] = chunk_id
    totals["candidate_rows"] = int(sessions["slate_size"].sum())
    (LOCAL_DIR / "step1_stats.json").write_text(json.dumps({k: int(v) for k, v in totals.items()}))
    del pend_c, pend_h, pend_s, sess_rows, needed, s_all, sessions
    gc.collect()
    mark_done("step1")
    backup("stage", "sessions.parquet", "arrays", "step1_stats.json", "step1.done")

stats1 = json.loads((LOCAL_DIR / "step1_stats.json").read_text())
sessions = pd.read_parquet(LOCAL_DIR / "sessions.parquet")
N_CHUNKS = int(stats1["chunks"])
print(f"Sessions in file:            {stats1['sessions']:,} (expected {EXPECTED['sessions']:,})")
print(f"Eligible sessions:           {stats1['eligible']:,} (expected {EXPECTED['eligible']:,})")
print(f"  excluded all-negative:     {stats1['all_negative']:,};  all-positive: {stats1['all_positive']:,}")
print(f"Candidate rows:              {stats1['candidate_rows']:,} (expected {EXPECTED['candidate_rows']:,})")
print(f"Chunks of {SESSIONS_PER_CHUNK} sessions:      {N_CHUNKS}")
print(f"Cart actions removed (>90d): {stats1['carts_removed']:,} (expected {EXPECTED['carts_removed']:,})")
print(f"Check - negative final gaps: {stats1['negative_last_gap'] / max(stats1['rows_with_times'], 1):.4%} of sessions (should be 0)")
print(f"Check - clicks within 7 days: {stats1['clicks_within_7d'] / max(stats1['clicks'], 1):.2%} (should be close to 100%)")
ram("after step 1")

Sessions in file:            173,831 (expected 173,831)
Eligible sessions:           171,175 (expected 171,175)
  excluded all-negative:     2,103;  all-positive: 553
Candidate rows:              15,348,049 (expected 15,348,049)
Chunks of 2000 sessions:      86
Cart actions removed (>90d): 1,177 (expected 1,177)
Check - negative final gaps: 0.0000% of sessions (should be 0)
Check - clicks within 7 days: 100.00% (should be close to 100%)
[RAM] after step 1: 1.13 GB in use


## Step 2 — Reproduce the train / validation / test split

**What this step does.** It applies Siqian's split unchanged. The eligible session IDs, in their original order, go through `train_test_split` with `test_size=0.20` and `random_state=42`. The held-out 20% is then split in half with the same seed. Because the input order, the sizes and the seed are identical, this reproduces `user_split.csv` exactly; Step 9 confirms that when the file exists.

**Why it matters.** The split decides which labels may be used to build family G (training sessions only). It also decides which sessions the evaluation in Step 8 may look at (training sessions only), so that validation and test sessions remain unseen until a model is evaluated on them.

In [7]:
from sklearn.model_selection import train_test_split

ids = pd.Series(sessions["session_id"].to_numpy(), name="session_id")
train_ids, temp_ids = train_test_split(ids, test_size=0.20, random_state=SEED, shuffle=True)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=SEED, shuffle=True)
split_map = pd.Series("train", index=ids.to_numpy())
split_map.loc[val_ids.to_numpy()] = "val"
split_map.loc[test_ids.to_numpy()] = "test"
sessions["split"] = split_map.reindex(sessions["session_id"].to_numpy()).to_numpy()
sessions.to_parquet(LOCAL_DIR / "sessions.parquet", index=False)
backup("sessions.parquet")
print(sessions["split"].value_counts())
del ids, train_ids, temp_ids, val_ids, test_ids

split
train    136940
test      17118
val       17117
Name: count, dtype: int64


## Step 3 — Product tables: attributes and title TF-IDF

**What this step does.** It reads the product metadata, keeping only the products that occur as a candidate or in a history (the set saved in Step 1), and builds two tables.

1. **Attribute table.** For each product: `brand_id`, `shop_id` and `cate_id_1` to `cate_id_4`, stored as 64-bit integers, with `-1` for a missing value. A `shop_id` of `-1` already means "no shop" in the data, so it is treated as missing too. A missing value never counts as a match in the features.
2. **Title TF-IDF matrix.** Each title (`name`) is split on `\x18` into term identifiers. Terms that appear in fewer than 5 titles are discarded, because they are too rare to link products. Each title becomes a vector whose entries are term frequency × inverse document frequency, with the smoothed IDF ln((1 + N) / (1 + df)) + 1 used by scikit-learn. Each vector is scaled to unit length, so that the dot product of two titles is their cosine similarity. Terms shared by very many products (generic words) receive low weights, and distinctive terms receive high weights.

Both tables are written to disk and later opened memory-mapped. The TF-IDF matrix is built one file at a time and appended to disk, so it never exists in full in RAM. Products are looked up through a sorted array of product IDs. If a product ID appears more than once in the metadata, the first occurrence is used, and the number of duplicates is printed (an open question in the proposal). One extra empty row at the end of both tables stands for "product not in the metadata".

**Source.** If Part 1's Parquet copy of the product file (`Raw Data/product_chunks/`) exists, it is read because it is faster; otherwise the text file is read in chunks. The contents are identical.

In [8]:
def product_chunks(columns):
    # Yield the product metadata in chunks, from the Parquet cache if present, else from the text file.
    cached = sorted(PRODUCT_CACHE_DIR.glob("*.parquet")) if PRODUCT_CACHE_DIR.exists() else []
    if cached:
        for f in cached:
            yield pd.read_parquet(f, columns=columns)
    else:
        for chunk in pd.read_csv(PRODUCT_TXT, sep="\t", dtype=str, usecols=columns, chunksize=500_000):
            yield chunk


def isin_sorted(values, sorted_ref):
    i = np.searchsorted(sorted_ref, values)
    i = np.minimum(i, len(sorted_ref) - 1)
    return sorted_ref[i] == values


if done("step3") and restore("arrays", "titles"):
    print("Step 3 already completed - skipping.")
else:
    needed = np.load(ARR_DIR / "needed_wids.npy")
    wid_parts, attr_parts, n_rows, n_bad = [], [], 0, 0
    for i, ch in enumerate(product_chunks(["wid", "name"] + ATTR_SOURCE)):
        w = pd.to_numeric(ch["wid"], errors="coerce")
        n_bad += int(w.isna().sum())
        keep = w.notna().to_numpy()
        keep[keep] = isin_sorted(w[keep].to_numpy(np.int64), needed)
        ch = ch[keep]
        wid_parts.append(w[keep].to_numpy(np.int64))
        a = np.column_stack([pd.to_numeric(ch[c], errors="coerce").fillna(-1).to_numpy(np.int64) for c in ATTR_SOURCE])
        attr_parts.append(a)
        ch[["name"]].fillna("").to_parquet(TITLE_DIR / f"titles_{i:03d}.parquet", index=False)
        n_rows += len(ch)
        print(f"product chunk {i}: kept {len(ch):,} rows (total {n_rows:,})", end="\r")
    prod_wid = np.concatenate(wid_parts); prod_attr = np.vstack(attr_parts + [np.full((1, 6), -1, np.int64)])
    del wid_parts, attr_parts
    N_PROD = len(prod_wid)
    # lookup: sorted unique IDs -> first row
    order = np.argsort(prod_wid, kind="stable")
    sw = prod_wid[order]
    first = np.ones(len(sw), bool); first[1:] = sw[1:] != sw[:-1]
    np.save(ARR_DIR / "lookup_wid.npy", sw[first]); np.save(ARR_DIR / "lookup_row.npy", order[first].astype(np.int64))
    np.save(ARR_DIR / "prod_attr.npy", prod_attr)
    meta = {"n_prod": int(N_PROD), "duplicates": int((~first).sum()), "non_integer_wid": n_bad,
            "needed": int(len(needed)), "needed_missing": int(len(needed) - first.sum())}
    del prod_wid, prod_attr, order, sw, first, needed
    gc.collect()

    # ---- title TF-IDF: pass 1 counts document frequencies, pass 2 writes the matrix ----
    title_files = sorted(TITLE_DIR.glob("titles_*.parquet"))
    df_count = Counter()
    for f in title_files:
        names = pd.read_parquet(f)["name"].to_numpy(dtype=object)
        df_count.update(chain.from_iterable(set(x.split(TERM_SEP)) - {""} for x in names))
    vocab = sorted(t for t, c in df_count.items() if c >= TITLE_MIN_DF)
    term_df = np.array([df_count[t] for t in vocab], dtype=np.float64)
    idf = np.log((1 + N_PROD) / (1 + term_df)) + 1.0
    vocab_index = pd.Index(vocab)
    pd.DataFrame({"term": vocab}).to_parquet(ARR_DIR / "title_vocab.parquet", index=False)
    del df_count, term_df

    indptr, offset = [np.zeros(1, np.int64)], 0
    with open(ARR_DIR / "title_indices.bin", "wb") as fi, open(ARR_DIR / "title_data.bin", "wb") as fd:
        for f in title_files:
            names = pd.read_parquet(f)["name"].to_numpy(dtype=object)
            toks, lens = split_flat(pd.Series(names), TERM_SEP)
            rows = np.repeat(np.arange(len(names)), lens)
            cols = vocab_index.get_indexer(toks)
            ok = cols >= 0
            m = sp.csr_matrix((np.ones(ok.sum(), np.float32), (rows[ok], cols[ok])), shape=(len(names), len(vocab)))
            m.sum_duplicates()                               # term frequencies (duplicates summed)
            m.data *= idf[m.indices].astype(np.float32)      # tf x idf
            m = normalize(m, norm="l2", copy=False)          # unit length: dot product = cosine
            m.indices.astype(np.int32).tofile(fi)
            m.data.astype(np.float32).tofile(fd)
            indptr.append(offset + m.indptr[1:].astype(np.int64))
            offset += int(m.nnz)
            del names, toks, lens, rows, cols, m
    indptr = np.concatenate(indptr)
    indptr = np.append(indptr, indptr[-1])                   # the extra empty row for unknown products
    np.save(ARR_DIR / "title_indptr.npy", indptr)
    meta.update({"vocab": len(vocab), "title_nnz": int(indptr[-1])})
    (ARR_DIR / "prod_meta.json").write_text(json.dumps(meta))
    del vocab, vocab_index, idf, indptr
    gc.collect()
    mark_done("step3")
    backup("arrays", "titles", "step3.done")

PROD_META = json.loads((ARR_DIR / "prod_meta.json").read_text())
print(PROD_META)
ram("after step 3")

{'n_prod': 11500314, 'duplicates': 0, 'non_integer_wid': 0, 'needed': 12223304, 'needed_missing': 722990, 'vocab': 368021, 'title_nnz': 268204961}
[RAM] after step 3: 1.99 GB in use


The helper functions below open the product tables memory-mapped and look up the rows of a set of product IDs. They are used by every later step. Unknown products map to the extra empty row, so their attributes are `-1` and their title vector is empty.

In [9]:
LOOKUP_WID = np.load(ARR_DIR / "lookup_wid.npy", mmap_mode="r")
LOOKUP_ROW = np.load(ARR_DIR / "lookup_row.npy", mmap_mode="r")
PROD_ATTR = np.load(ARR_DIR / "prod_attr.npy", mmap_mode="r")
EMPTY_ROW = PROD_META["n_prod"]
TITLE_VOCAB = pd.read_parquet(ARR_DIR / "title_vocab.parquet")["term"]
TITLE_INDEX = pd.Index(TITLE_VOCAB)


def prod_rows(wids):
    # Row of each product ID in the product tables (EMPTY_ROW if the product has no metadata).
    wids = np.asarray(wids, dtype=np.int64)
    i = np.searchsorted(LOOKUP_WID, wids)
    i_c = np.minimum(i, len(LOOKUP_WID) - 1)
    found = np.asarray(LOOKUP_WID[i_c]) == wids
    return np.where(found, np.asarray(LOOKUP_ROW[i_c]), EMPTY_ROW)


def open_title_matrix():
    indptr = np.load(ARR_DIR / "title_indptr.npy", mmap_mode="r")
    nnz = int(indptr[-1])
    indices = np.memmap(ARR_DIR / "title_indices.bin", dtype=np.int32, mode="r", shape=(nnz,))
    data = np.memmap(ARR_DIR / "title_data.bin", dtype=np.float32, mode="r", shape=(nnz,))
    return sp.csr_matrix((data, indices, np.asarray(indptr)), shape=(len(indptr) - 1, len(TITLE_VOCAB)))


TITLES = open_title_matrix()
print("title matrix:", TITLES.shape, "non-zeros:", TITLES.nnz)

title matrix: (11500315, 368021) non-zeros: 268204961


## Step 4 — Embeddings: item2vec on histories, term2vec on titles

**What this step does.** It trains two word2vec models with `gensim`. Both use 64 dimensions, a context window of 5, skip-gram and 5 epochs.

1. **Item2vec.** Each eligible session's cleaned history is treated as a "sentence" whose "words" are product IDs, oldest first. Products that occur in similar contexts in users' histories (bought or clicked around the same other products) receive nearby vectors. This captures substitutes and complements that share no attribute or title term, such as a phone and its case. Products occurring fewer than 5 times across all histories get no vector, because their vectors would be unreliable. Histories of all splits are used: they precede the query and contain no labels. Candidate lists and labels are never used.
2. **Term2vec.** Each product title is a sentence of term IDs. The vector of a product's title is the TF-IDF-weighted average of its term vectors, scaled to unit length. This gives a vector to every product with a title, including products that never occur in a history, which is where item2vec cannot help.

The training data are streamed from the chunk files on disk for every epoch, so the histories are never held in memory. After training, the vectors are scaled to unit length and saved memory-mapped, and the gensim models are deleted. The item vectors are stored in a sorted array of product IDs; the term vectors are aligned with the TF-IDF vocabulary of Step 3. Training uses all CPU cores, so the vectors can differ slightly between runs.

In [10]:
from gensim.models import Word2Vec


class HistoryCorpus:
    # Re-iterable corpus: one sentence (list of product-ID strings) per session, streamed from disk.
    def __iter__(self):
        for i in range(N_CHUNKS):
            h = pd.read_parquet(STAGE_DIR / f"hist_{i:03d}.parquet", columns=["session_id", "wid"])
            sid = h["session_id"].to_numpy()
            toks = h["wid"].to_numpy().astype(str)
            cuts = np.flatnonzero(np.diff(sid)) + 1
            for sent in np.split(toks, cuts):
                if len(sent):
                    yield sent.tolist()


class TitleCorpus:
    # Re-iterable corpus: one sentence (list of term IDs) per product title, streamed from disk.
    def __iter__(self):
        for f in sorted(TITLE_DIR.glob("titles_*.parquet")):
            for name in pd.read_parquet(f)["name"].to_numpy(dtype=object):
                if name:
                    yield name.split(TERM_SEP)


if done("step4") and restore("arrays"):
    print("Step 4 already completed - skipping.")
else:
    workers = max(1, os.cpu_count() or 1)
    t0 = time.time()
    i2v = Word2Vec(HistoryCorpus(), vector_size=EMB_DIM, window=EMB_WINDOW, min_count=I2V_MIN_COUNT, sg=1,
                   epochs=EMB_EPOCHS, workers=workers, seed=SEED)
    keys = np.array(i2v.wv.index_to_key, dtype=np.int64)
    vecs = normalize(i2v.wv.vectors).astype(np.float32)
    order = np.argsort(keys)
    np.save(ARR_DIR / "i2v_keys.npy", keys[order]); np.save(ARR_DIR / "i2v_vecs.npy", vecs[order])
    print(f"item2vec: {len(keys):,} products with vectors ({time.time() - t0:.0f}s)")
    del i2v, keys, vecs, order; gc.collect()

    t0 = time.time()
    t2v = Word2Vec(TitleCorpus(), vector_size=EMB_DIM, window=EMB_WINDOW, min_count=TITLE_MIN_DF, sg=1,
                   epochs=EMB_EPOCHS, workers=workers, seed=SEED)
    E = np.zeros((len(TITLE_VOCAB), EMB_DIM), np.float32)
    k2i = t2v.wv.key_to_index
    pos = np.array([k2i.get(t, -1) for t in TITLE_VOCAB])
    E[pos >= 0] = t2v.wv.vectors[pos[pos >= 0]]
    np.save(ARR_DIR / "t2v_term_vecs.npy", E)
    print(f"term2vec: {(pos >= 0).sum():,} of {len(TITLE_VOCAB):,} title terms with vectors ({time.time() - t0:.0f}s)")
    del t2v, E, k2i, pos; gc.collect()
    mark_done("step4")
    backup("arrays", "step4.done")

I2V_KEYS = np.load(ARR_DIR / "i2v_keys.npy", mmap_mode="r")
I2V_VECS = np.load(ARR_DIR / "i2v_vecs.npy", mmap_mode="r")
T2V_TERMS = np.load(ARR_DIR / "t2v_term_vecs.npy", mmap_mode="r")


def i2v_rows(wids):
    # Index of each product's item2vec vector, or -1 if it has none.
    wids = np.asarray(wids, dtype=np.int64)
    if len(I2V_KEYS) == 0:
        return np.full(len(wids), -1)
    i = np.minimum(np.searchsorted(I2V_KEYS, wids), len(I2V_KEYS) - 1)
    return np.where(np.asarray(I2V_KEYS[i]) == wids, i, -1)


print("item vectors:", I2V_VECS.shape, " term vectors:", T2V_TERMS.shape)
ram("after step 4")

item2vec: 859,412 products with vectors (216s)
term2vec: 368,021 of 368,021 title terms with vectors (1569s)
item vectors: (859412, 64)  term vectors: (368021, 64)
[RAM] after step 4: 1.93 GB in use


## Step 5 — Training-set priors (family G), computed out-of-fold

**What this step does.** It computes, for every candidate row, how often products like it were engaged with in *training* sessions. This is the only family that uses labels. The smoothed engagement rate of a value *a* (a product, a brand or a level-3 category) is

> ctr(a) = (pos(a) + α·p₀) / (n(a) + α), with α = 20,

where n(a) is the number of training rows with value *a*, pos(a) is the number of those with a label above zero, and p₀ is the overall positive rate in training. The smoothing pulls rarely shown values towards p₀, so that a product shown once and clicked once does not receive a rate of 100%.

**How leakage is avoided.**
- Validation and test rows receive rates computed from all training sessions. Their own labels are never used.
- Training rows receive rates computed **out-of-fold**. The training sessions are divided into 5 folds, and a row's rate excludes the fold containing its own session. Otherwise the feature would contain the row's own label, which is the flaw of Part 2's full-data `pop_score`, and the model would learn to read the answer from it.
- `item_impr_train` is log(1 + the number of training rows showing the product). It uses no labels.

A missing brand or category gives a missing rate. The table of all candidate keys (about 15 million rows of integers, roughly 0.3 GB) is held in memory only for this step, and the results are written per chunk.

In [11]:
if done("step5") and restore("stage"):
    print("Step 5 already completed - skipping.")
else:
    parts = []
    for i in range(N_CHUNKS):
        c = pd.read_parquet(STAGE_DIR / f"cand_{i:03d}.parquet", columns=["session_id", "wid", "label"])
        a = PROD_ATTR[prod_rows(c["wid"].to_numpy())]
        c["brand"] = a[:, BRAND]; c["cate3"] = a[:, CATE3]
        c["chunk"] = np.int16(i)
        parts.append(c)
    cand_all = pd.concat(parts, ignore_index=True); del parts
    split_of = sessions.set_index("session_id")["split"]
    cand_all["split"] = pd.Categorical(split_of.reindex(cand_all["session_id"].to_numpy()).to_numpy())
    cand_all["pos"] = (cand_all["label"] > 0).astype(np.int8)

    # folds over training sessions
    tr_sess = sessions.loc[sessions["split"] == "train", "session_id"].to_numpy()
    fold_of = pd.Series(np.random.default_rng(SEED).permutation(len(tr_sess)) % N_FOLDS, index=tr_sess)
    is_tr = (cand_all["split"] == "train").to_numpy()
    cand_all["fold"] = np.int8(-1)
    cand_all.loc[is_tr, "fold"] = fold_of.reindex(cand_all.loc[is_tr, "session_id"].to_numpy()).to_numpy()
    tr = cand_all[is_tr]
    p0 = tr["pos"].mean()
    print(f"training rows: {len(tr):,}; positive rate p0 = {p0:.4f}")

    for key, col in [("wid", "item_ctr_train"), ("brand", "brand_ctr_train"), ("cate3", "cate3_ctr_train")]:
        full = tr.groupby(key)["pos"].agg(["size", "sum"])
        by_fold = tr.groupby([key, "fold"])["pos"].agg(["size", "sum"])
        keys = cand_all[key].to_numpy()
        n_full = full["size"].reindex(keys).fillna(0).to_numpy()
        p_full = full["sum"].reindex(keys).fillna(0).to_numpy()
        idx = pd.MultiIndex.from_arrays([keys[is_tr], cand_all.loc[is_tr, "fold"].to_numpy()])
        n_full[is_tr] -= by_fold["size"].reindex(idx).fillna(0).to_numpy()   # leave own fold out
        p_full[is_tr] -= by_fold["sum"].reindex(idx).fillna(0).to_numpy()
        rate = (p_full + PRIOR_ALPHA * p0) / (n_full + PRIOR_ALPHA)
        if key != "wid":
            rate = np.where(keys == -1, np.nan, rate)
        cand_all[col] = rate.astype(np.float32)
        if key == "wid":
            cand_all["item_impr_train"] = np.log1p(full["size"].reindex(keys).fillna(0).to_numpy()).astype(np.float32)
        del full, by_fold, keys, n_full, p_full, idx, rate
    for i, g in cand_all.groupby("chunk", sort=True):
        g[["session_id", "wid"] + FAMILIES["G"]].to_parquet(STAGE_DIR / f"priors_{i:03d}.parquet", index=False)
    del cand_all, tr, fold_of
    gc.collect()
    mark_done("step5")
    backup("stage", "step5.done")
ram("after step 5")

/tmp/ipykernel_1527/3872964315.py:21: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[2 2 2 ... 0 0 0]' has dtype incompatible with int8, please explicitly cast to a compatible dtype first.
  cand_all.loc[is_tr, "fold"] = fold_of.reindex(cand_all.loc[is_tr, "session_id"].to_numpy()).to_numpy()


training rows: 12,279,873; positive rate p0 = 0.0209
[RAM] after step 5: 3.62 GB in use


## Step 6 — Feature computation by chunk

**What this step does.** For each of the 86 chunks, it reads the chunk's candidates, cleaned histories and priors. It looks up the products' attributes, titles and vectors from the memory-mapped tables, computes all 90 features, and writes `feature_chunks/feat_XXX.parquet` to Drive. The rows are in the same order as the chunk's candidate file. Features are stored as 32-bit floats. A missing value (`NaN`) means "no evidence", for example a brand the user has never interacted with, and is deliberately not replaced by zero (see the proposal, Section 4).

Every past action carries a weight **w = type weight × exp(−elapsed days / 30)**, with type weights CLICK 1, FLW 2, CART 3 and ORD 5. Recent and stronger actions therefore count for more.

The families, in the order they are computed:

- **Context (session level).** Part 1's 12 counts (each action type in the windows 0–7, 7–90 and 90–365 days), the history length, the age of the oldest action (`history_span_days`) and of the newest (`days_since_last_interaction`). Then the share of each action type, the entropy of the level-3 categories in the history (low means a focused user), the number of distinct brands, the share of actions reached without a query (`-1`), and `burst_len`, the number of actions in the final run of actions less than 30 minutes apart. These describe the user, not the candidate. They let the model decide how far to rely on the similarity families, for example more for a long, focused history than for a user with three actions.
- **A. Exact product** (Part 1 definitions). The counts of the candidate's own product in the history by type and window, and the elapsed days since it was last seen. Unmatched rows have counts of 0 and a missing last-seen value, as in Part 1.
- **B. Attribute affinity.** For the brand, the shop and each category level: the summed weight of history actions with the candidate's value (`_aff`); that weight as a share of the session's total weight (`_share`); the days since the value was last seen; and the number of orders with that value.
- **C. Title similarity.** The cosine between the candidate's TF-IDF title and the user's title profile (the weight-summed history titles, scaled to unit length); the same using ordered products only; the largest cosine with any single history title; and the share of the candidate's title terms that appear in the user's past queries.
- **D. Embedding similarity.** The candidate's item2vec vector compared with the history's vectors: the mean cosine, the maximum cosine, and the cosine with the mean of the five most recent known vectors. Also the cosine between the candidate's term2vec title vector and the mean of the history's title vectors.
- **E. Recency of interest.** Whether the candidate shares the brand or the level-3 category of the most recent action, and its title and item2vec cosines with that action.
- **F. Query-conditioned history.** Using only the history actions whose past query shares at least one term with the test query: their number, how many were on the candidate's own product, and the affinity for the candidate's brand and level-3 category.
- **G. Training-set priors** from Step 5.
- **H. Position within the slate.** For seven key features, the percentile rank (1 is highest) and the z-score among the candidates of the same session. Also the share of the slate sharing the candidate's level-3 category, and the slate size. Because the ranker compares candidates within a slate, these relative values transfer between users better than absolute ones.

Most operations are vectorised over the whole chunk. Only the "maximum similarity with any history item" (`title_cos_max`, `i2v_cos_max`) needs a small loop over sessions, because it compares every candidate with every history item of the same session.

In [12]:
def group_bounds(sorted_local, n_groups):
    # Start and end row of each group in an array sorted by group.
    starts = np.searchsorted(sorted_local, np.arange(n_groups), side="left")
    ends = np.searchsorted(sorted_local, np.arange(n_groups), side="right")
    return starts, ends


def rowdot(A, B):
    # Row-wise dot product of two sparse matrices with the same shape.
    return np.asarray(A.multiply(B).sum(axis=1)).ravel()


def nan_if(values, mask):
    out = np.asarray(values, dtype=np.float64).copy()
    out[mask] = np.nan
    return out


def keyed_merge(c_keys, h_keys, h_values, how="sum"):
    # For each candidate key (session, value), aggregate h_values over history rows with the same key.
    # Keys are 2-column int64 arrays; returns NaN where no history row matches.
    h = pd.DataFrame({"s": h_keys[0], "v": h_keys[1], "x": h_values})
    agg = h.groupby(["s", "v"])["x"].agg(how)
    idx = pd.MultiIndex.from_arrays([c_keys[0], c_keys[1]])
    return agg.reindex(idx).to_numpy(dtype=np.float64)


def build_chunk_features(i):
    cand = pd.read_parquet(STAGE_DIR / f"cand_{i:03d}.parquet")
    hist = pd.read_parquet(STAGE_DIR / f"hist_{i:03d}.parquet")
    pri = pd.read_parquet(STAGE_DIR / f"priors_{i:03d}.parquet")
    assert (pri["session_id"].to_numpy() == cand["session_id"].to_numpy()).all()
    assert (pri["wid"].to_numpy() == cand["wid"].to_numpy()).all()
    sess = sessions[sessions["chunk"] == i].sort_values("session_id")
    sids = sess["session_id"].to_numpy()
    nS = len(sids)
    c_loc = np.searchsorted(sids, cand["session_id"].to_numpy())
    h_loc = np.searchsorted(sids, hist["session_id"].to_numpy())
    nC, nH = len(cand), len(hist)
    F = {}

    c_wid, h_wid = cand["wid"].to_numpy(), hist["wid"].to_numpy()
    c_row, h_row = prod_rows(c_wid), prod_rows(h_wid)
    c_attr, h_attr = np.asarray(PROD_ATTR[c_row]), np.asarray(PROD_ATTR[h_row])
    typ = hist["type"].to_numpy().astype(np.int64)
    days = hist["days"].to_numpy().astype(np.float64)
    w = np.where(typ >= 0, TYPE_WEIGHT[np.clip(typ, 0, 3)], 1.0) * np.exp(-days / TAU_DAYS)
    tot_w = np.bincount(h_loc, weights=w, minlength=nS)
    h_len = np.bincount(h_loc, minlength=nS)
    h_start, h_end = group_bounds(h_loc, nS)
    c_start, c_end = group_bounds(c_loc, nS)
    has_h = h_len > 0
    last_idx = np.where(has_h, h_end - 1, 0)

    # ---------------- context (session level, broadcast to candidates) ----------------
    ctx = {}
    for t, tname in enumerate(TYPE_NAMES):
        for (a, b), wname in zip([(0, 7), (7, 90), (90, 365)], ["0_7d", "7_90d", "90_365d"]):
            m = (typ == t) & (days >= a) & (days < b)
            ctx[f"{tname}_count_{wname}"] = np.bincount(h_loc[m], minlength=nS)
    ctx["history_len_clean"] = h_len
    dser = pd.Series(days).groupby(h_loc)
    ctx["history_span_days"] = dser.max().reindex(range(nS)).to_numpy()
    ctx["days_since_last_interaction"] = dser.min().reindex(range(nS)).to_numpy()
    for t, tname in enumerate(TYPE_NAMES):
        ctx[f"share_{tname}"] = nan_if(np.bincount(h_loc[typ == t], minlength=nS) / np.maximum(h_len, 1), ~has_h)
    c3 = h_attr[:, CATE3]
    m3 = c3 != -1
    cc = pd.DataFrame({"s": h_loc[m3], "v": c3[m3]}).value_counts()
    p = cc / cc.groupby(level=0).transform("sum")
    ent = (-(p * np.log(p))).groupby(level=0).sum()
    ctx["cate3_entropy"] = nan_if(ent.reindex(range(nS)).fillna(0).to_numpy(), ~has_h)
    br = h_attr[:, BRAND]
    ctx["n_distinct_brands"] = pd.DataFrame({"s": h_loc[br != -1], "v": br[br != -1]}).drop_duplicates()["s"] \
        .value_counts().reindex(range(nS)).fillna(0).to_numpy()
    qryless = (hist["hqry"].to_numpy(dtype=object) == "-1")
    ctx["qryless_share"] = nan_if(np.bincount(h_loc[qryless], minlength=nS) / np.maximum(h_len, 1), ~has_h)
    same_next = np.zeros(nH, bool); same_next[:-1] = h_loc[:-1] == h_loc[1:]
    gap_next = np.zeros(nH); gap_next[:-1] = days[:-1] - days[1:]
    brk = same_next & (gap_next >= BURST_GAP_DAYS)
    hpos = np.arange(nH) - h_start[h_loc]
    last_brk = np.full(nS, -1); np.maximum.at(last_brk, h_loc[brk], hpos[brk])
    ctx["burst_len"] = h_len - (last_brk + 1)
    for k in FAMILIES["context"]:
        F[k] = np.asarray(ctx[k], dtype=np.float64)[c_loc]

    # ---------------- A. exact product (Part 1) ----------------
    defs = [("exact_wid_click_0_7d", 0, 0, 7), ("exact_wid_cart_0_7d", 1, 0, 7), ("exact_wid_ord_0_7d", 2, 0, 7),
            ("exact_wid_cart_7_90d", 1, 7, 90), ("exact_wid_ord_7_90d", 2, 7, 90), ("exact_wid_ord_90_365d", 2, 90, 365)]
    hdf = pd.DataFrame({"s": h_loc, "v": h_wid, "d": days})
    for name, t, a, b in defs:
        hdf[name] = ((typ == t) & (days >= a) & (days < b)).astype(np.int32)
    g = hdf.groupby(["s", "v"]).agg({**{n: "sum" for n, *_ in defs}, "d": "min"})
    idx = pd.MultiIndex.from_arrays([c_loc, c_wid])
    ga = g.reindex(idx)
    for name, *_ in defs:
        F[name] = ga[name].fillna(0).to_numpy()
    F["exact_wid_last_seen_days"] = ga["d"].to_numpy()
    del hdf, g, ga

    # ---------------- B. attribute affinity ----------------
    is_ord = (typ == TYPE_CODE["ORD"]).astype(np.float64)
    for j, k in enumerate(ATTRS):
        hv, cv = h_attr[:, j], c_attr[:, j]
        hm = hv != -1
        keys_h = (h_loc[hm], hv[hm])
        keys_c = (c_loc, cv)
        aff = keyed_merge(keys_c, keys_h, w[hm], "sum")
        last = keyed_merge(keys_c, keys_h, days[hm], "min")
        ordc = keyed_merge(keys_c, keys_h, is_ord[hm], "sum")
        miss = cv == -1
        F[f"{k}_aff"] = nan_if(aff, miss)
        F[f"{k}_share"] = nan_if(aff / np.maximum(tot_w[c_loc], 1e-12), miss)
        F[f"{k}_last_seen_days"] = nan_if(last, miss)
        F[f"{k}_ord_cnt"] = nan_if(ordc, miss)

    # ---------------- C. title similarity ----------------
    Xc, Xh = TITLES[c_row], TITLES[h_row]
    c_nnz = np.diff(Xc.indptr)
    S = sp.csr_matrix((w, (h_loc, np.arange(nH))), shape=(nS, nH))
    P = normalize(S @ Xh)
    p_nnz = np.diff(P.indptr)
    F["title_cos_profile"] = nan_if(rowdot(P[c_loc], Xc), (c_nnz == 0) | (p_nnz[c_loc] == 0))
    S_ord = sp.csr_matrix((w * is_ord, (h_loc, np.arange(nH))), shape=(nS, nH))
    P_ord = normalize(S_ord @ Xh)
    po_nnz = np.diff(P_ord.indptr)
    F["title_cos_profile_ord"] = nan_if(rowdot(P_ord[c_loc], Xc), (c_nnz == 0) | (po_nnz[c_loc] == 0))
    Xlast = Xh[last_idx]
    l_nnz = np.diff(Xlast.indptr)
    F["last_title_cos"] = nan_if(rowdot(Xlast[c_loc], Xc), (c_nnz == 0) | (l_nnz[c_loc] == 0) | ~has_h[c_loc])
    # overlap of candidate title terms with the user's past query terms
    hq = hist["hqry"].to_numpy(dtype=object)
    qm = ~qryless
    q_toks, q_lens = split_flat(pd.Series(hq[qm]), TERM_SEP)
    q_cols = TITLE_INDEX.get_indexer(q_toks)
    q_rows = np.repeat(h_loc[qm], q_lens)
    ok = q_cols >= 0
    Q = sp.csr_matrix((np.ones(ok.sum(), np.float32), (q_rows[ok], q_cols[ok])), shape=(nS, len(TITLE_VOCAB)))
    Q.data[:] = 1.0                                            # term present (duplicates collapsed)
    Bc = Xc.copy(); Bc.data[:] = 1.0
    F["title_hist_qry_overlap"] = nan_if(rowdot(Q[c_loc], Bc) / np.maximum(c_nnz, 1), c_nnz == 0)

    # ---------------- D. embedding similarity ----------------
    c_vi, h_vi = i2v_rows(c_wid), i2v_rows(h_wid)
    Vc = np.zeros((nC, EMB_DIM), np.float32); Vc[c_vi >= 0] = I2V_VECS[c_vi[c_vi >= 0]]
    hk = h_vi >= 0
    Vh = np.zeros((nH, EMB_DIM), np.float32); Vh[hk] = I2V_VECS[h_vi[hk]]
    n_known = np.bincount(h_loc[hk], minlength=nS)

    def session_sum(mask, M):
        # Sum the rows of M selected by mask within each session (a sparse indicator product, no Python loop).
        A = sp.csr_matrix((np.ones(mask.sum()), (h_loc[mask], np.flatnonzero(mask))), shape=(nS, nH))
        return np.asarray(A @ M)

    mean_v = session_sum(hk, Vh) / np.maximum(n_known, 1)[:, None]
    c_unknown = c_vi < 0
    F["i2v_cos_mean"] = nan_if((Vc * mean_v[c_loc]).sum(1), c_unknown | (n_known[c_loc] == 0))
    cum = np.cumsum(hk)
    within = cum - np.r_[0, cum][h_start][h_loc]              # known vectors up to and including this row
    rank_from_end = n_known[h_loc] - within + hk               # 1 = most recent known vector of the session
    last5 = hk & (rank_from_end <= 5)
    l5 = normalize(session_sum(last5, Vh))
    F["i2v_cos_last5"] = nan_if((Vc * l5[c_loc]).sum(1), c_unknown | (n_known[c_loc] == 0))
    F["last_i2v_cos"] = nan_if((Vc * Vh[last_idx][c_loc]).sum(1), c_unknown | ~has_h[c_loc] | ~hk[last_idx][c_loc])
    Tc = normalize(np.asarray(Xc @ T2V_TERMS)); Th = normalize(np.asarray(Xh @ T2V_TERMS))
    th_ok = np.abs(Th).sum(1) > 0
    tprof = normalize(session_sum(th_ok, Th))
    t_nn = np.abs(tprof).sum(1) > 0
    F["t2v_cos_profile"] = nan_if((Tc * tprof[c_loc]).sum(1), (np.abs(Tc).sum(1) == 0) | ~t_nn[c_loc])

    # maximum similarity with any single history item (loop over sessions)
    tmax = np.full(nC, np.nan); imax = np.full(nC, np.nan)
    for s in range(nS):
        a, b, c0, d0 = c_start[s], c_end[s], h_start[s], h_end[s]
        if b == a or d0 == c0:
            continue
        M = (Xc[a:b] @ Xh[c0:d0].T).toarray()
        tmax[a:b] = M.max(axis=1)
        kh = hk[c0:d0]
        if kh.any():
            imax[a:b] = (Vc[a:b] @ Vh[c0:d0][kh].T).max(axis=1)
    F["title_cos_max"] = nan_if(tmax, c_nnz == 0)
    F["i2v_cos_max"] = nan_if(imax, c_unknown)

    # ---------------- E. recency of interest ----------------
    for name, j in [("last_same_brand", BRAND), ("last_same_cate3", CATE3)]:
        lv = h_attr[last_idx, j][c_loc]
        cv = c_attr[:, j]
        F[name] = nan_if((cv == lv).astype(np.float64), (cv == -1) | (lv == -1) | ~has_h[c_loc])

    # ---------------- F. query-conditioned history ----------------
    q_terms = sess["query"].to_numpy(dtype=object)
    qt, ql = split_flat(pd.Series(q_terms), TERM_SEP)
    query_long = pd.DataFrame({"s": np.repeat(np.arange(nS), ql), "term": qt}).drop_duplicates()
    hq_long = pd.DataFrame({"r": np.repeat(np.flatnonzero(qm), q_lens), "s": q_rows, "term": q_toks})
    matched_rows = hq_long.merge(query_long, on=["s", "term"])["r"].unique()
    mt = np.zeros(nH, bool); mt[matched_rows] = True
    F["qhist_n_actions"] = np.bincount(h_loc[mt], minlength=nS)[c_loc].astype(np.float64)
    F["qhist_wid_cnt"] = np.nan_to_num(keyed_merge((c_loc, c_wid), (h_loc[mt], h_wid[mt]), np.ones(mt.sum()), "sum"))
    for name, j in [("qhist_brand_aff", BRAND), ("qhist_cate3_aff", CATE3)]:
        hv = h_attr[:, j]; mm = mt & (hv != -1)
        F[name] = nan_if(keyed_merge((c_loc, c_attr[:, j]), (h_loc[mm], hv[mm]), w[mm], "sum"), c_attr[:, j] == -1)

    # ---------------- G. training-set priors ----------------
    for k in FAMILIES["G"]:
        F[k] = pri[k].to_numpy(dtype=np.float64)

    # ---------------- H. position within the slate ----------------
    for f in H_BASE:
        x = pd.Series(F[f])
        grp = x.groupby(c_loc)
        F[f + "_slate_pct"] = grp.rank(ascending=False, pct=True).to_numpy()
        sd = grp.transform("std").replace(0, np.nan)
        F[f + "_slate_z"] = ((x - grp.transform("mean")) / sd).to_numpy()
    slate = np.bincount(c_loc, minlength=nS)
    F["slate_size"] = slate[c_loc].astype(np.float64)
    cv3 = c_attr[:, CATE3]
    same3 = pd.Series(1, index=pd.MultiIndex.from_arrays([c_loc, cv3])).groupby(level=[0, 1]).transform("sum").to_numpy()
    F["cate3_slate_share"] = nan_if(same3 / slate[c_loc], cv3 == -1)

    # ---------------- assemble ----------------
    out = pd.DataFrame({
        "session_id": cand["session_id"].to_numpy(),
        "wid": c_wid,
        "label": cand["label"].to_numpy(),
        "split": sess["split"].to_numpy()[c_loc],
    })
    feats = pd.DataFrame({k: np.asarray(F[k], dtype=np.float32) for k in FEATURE_COLUMNS})
    return pd.concat([out, feats], axis=1)

In [13]:
t_start = time.time()
for i in range(N_CHUNKS):
    out_file = FEAT_DIR / f"feat_{i:03d}.parquet"
    if out_file.exists():
        continue
    t0 = time.time()
    feats = build_chunk_features(i)
    assert list(feats.columns) == ID_COLUMNS + FEATURE_COLUMNS
    feats.to_parquet(out_file, index=False)
    print(f"chunk {i + 1}/{N_CHUNKS}: {len(feats):,} rows in {time.time() - t0:.0f}s "
          f"(elapsed {(time.time() - t_start) / 60:.1f} min)")
    del feats
    gc.collect()
    if i % 10 == 0:
        ram(f"after chunk {i}")
n_files = len(list(FEAT_DIR.glob("feat_*.parquet")))
print(f"Feature files on Drive: {n_files} of {N_CHUNKS}")

chunk 1/86: 179,397 rows in 11s (elapsed 0.2 min)
[RAM] after chunk 0: 6.14 GB in use
chunk 2/86: 183,012 rows in 11s (elapsed 0.4 min)
chunk 3/86: 176,513 rows in 10s (elapsed 0.5 min)
chunk 4/86: 179,332 rows in 11s (elapsed 0.7 min)
chunk 5/86: 178,637 rows in 10s (elapsed 0.9 min)
chunk 6/86: 177,387 rows in 11s (elapsed 1.1 min)
chunk 7/86: 177,790 rows in 10s (elapsed 1.2 min)
chunk 8/86: 180,005 rows in 11s (elapsed 1.4 min)
chunk 9/86: 177,530 rows in 10s (elapsed 1.6 min)
chunk 10/86: 177,475 rows in 11s (elapsed 1.8 min)
chunk 11/86: 178,821 rows in 10s (elapsed 2.0 min)
[RAM] after chunk 10: 6.17 GB in use
chunk 12/86: 177,772 rows in 10s (elapsed 2.1 min)
chunk 13/86: 182,592 rows in 11s (elapsed 2.3 min)
chunk 14/86: 181,554 rows in 11s (elapsed 2.5 min)
chunk 15/86: 176,980 rows in 11s (elapsed 2.7 min)
chunk 16/86: 177,250 rows in 10s (elapsed 2.8 min)
chunk 17/86: 181,853 rows in 11s (elapsed 3.0 min)
chunk 18/86: 178,183 rows in 11s (elapsed 3.2 min)
chunk 19/86: 179,3

## Step 7 — Feature dictionary

**What this step does.** It writes `feature_dictionary_part3.csv`, with one row per feature: its family, its definition and the meaning of a missing value. Together with the proposal, it documents what each column of the feature files means.

In [14]:
DESCR = {
    "exact_wid_click_0_7d": "Clicks on this exact product 0-7 days before the query (Part 1)",
    "exact_wid_cart_0_7d": "Cart additions of this exact product 0-7 days before the query (Part 1)",
    "exact_wid_ord_0_7d": "Orders of this exact product 0-7 days before the query (Part 1)",
    "exact_wid_cart_7_90d": "Cart additions of this exact product 7-90 days before the query (Part 1)",
    "exact_wid_ord_7_90d": "Orders of this exact product 7-90 days before the query (Part 1)",
    "exact_wid_ord_90_365d": "Orders of this exact product 90-365 days before the query (Part 1)",
    "exact_wid_last_seen_days": "Days since the most recent action on this exact product; missing if never seen (Part 1)",
    "title_cos_profile": "Cosine of the candidate's TF-IDF title with the user's weighted title profile",
    "title_cos_profile_ord": "As title_cos_profile, using ordered products only; missing if no orders",
    "title_cos_max": "Largest title cosine with any single history product",
    "title_hist_qry_overlap": "Share of the candidate's title terms that occur in the user's past queries",
    "i2v_cos_mean": "Mean item2vec cosine with the history products that have vectors",
    "i2v_cos_max": "Largest item2vec cosine with any history product",
    "i2v_cos_last5": "Item2vec cosine with the mean of the five most recent history vectors",
    "t2v_cos_profile": "Cosine of the candidate's term2vec title vector with the mean history title vector",
    "last_same_brand": "1 if the candidate has the brand of the most recent action, else 0",
    "last_same_cate3": "1 if the candidate has the level-3 category of the most recent action, else 0",
    "last_title_cos": "Title cosine with the product of the most recent action",
    "last_i2v_cos": "Item2vec cosine with the product of the most recent action",
    "qhist_n_actions": "Number of history actions whose past query shares a term with the test query (session level)",
    "qhist_wid_cnt": "Number of those query-matched actions on the candidate's own product",
    "qhist_brand_aff": "Weighted affinity for the candidate's brand among query-matched actions",
    "qhist_cate3_aff": "Weighted affinity for the candidate's level-3 category among query-matched actions",
    "item_ctr_train": "Smoothed engagement rate of the product in training sessions (out-of-fold for training rows)",
    "brand_ctr_train": "Smoothed engagement rate of the brand in training sessions (out-of-fold for training rows)",
    "cate3_ctr_train": "Smoothed engagement rate of the level-3 category in training sessions (out-of-fold)",
    "item_impr_train": "log(1 + number of training rows showing the product)",
    "cate3_slate_share": "Share of the slate's candidates with the candidate's level-3 category",
    "slate_size": "Number of candidates in the session",
    "history_len_clean": "Number of history actions after cleaning (Part 1)",
    "history_span_days": "Days since the oldest history action (Part 1)",
    "days_since_last_interaction": "Days since the most recent history action (Part 1)",
    "cate3_entropy": "Entropy of the level-3 categories in the history (low = focused user)",
    "n_distinct_brands": "Number of distinct brands in the history",
    "qryless_share": "Share of history actions not preceded by a query (-1)",
    "burst_len": "Number of actions in the final run of actions less than 30 minutes apart",
}
for k, kn in zip(ATTRS, ["brand", "shop", "level-1 category", "level-2 category", "level-3 category", "level-4 category"]):
    DESCR[f"{k}_aff"] = f"Recency- and action-weighted sum of history actions with the candidate's {kn}"
    DESCR[f"{k}_share"] = f"{k}_aff as a share of the session's total history weight"
    DESCR[f"{k}_last_seen_days"] = f"Days since the candidate's {kn} last occurred in the history"
    DESCR[f"{k}_ord_cnt"] = f"Number of orders of products with the candidate's {kn}"
for t in TYPE_NAMES:
    DESCR[f"share_{t}"] = f"Share of history actions of type {t.upper()}"
    for wname in ["0_7d", "7_90d", "90_365d"]:
        DESCR[f"{t}_count_{wname}"] = f"Number of {t.upper()} actions {wname.replace('_', '-').replace('d', ' days')} before the query (Part 1)"
for f in H_BASE:
    DESCR[f + "_slate_pct"] = f"Percentile rank of {f} among the session's candidates (1 = highest)"
    DESCR[f + "_slate_z"] = f"z-score of {f} among the session's candidates"

def missing_meaning(f):
    fam = FAMILY_OF[f]
    if f.startswith("exact_wid") and f != "exact_wid_last_seen_days" or f in ("qhist_wid_cnt", "qhist_n_actions", "slate_size"):
        return "never missing (0 = none)"
    if fam == "context":
        return "missing only if the session has no history"
    if fam == "G":
        return "missing if the brand or category is unknown"
    if fam == "H":
        return "missing if the underlying feature is missing or the slate has one candidate"
    return "missing = no evidence (value never seen in the history, or no title / vector)"

assert set(DESCR) == set(FEATURE_COLUMNS), set(FEATURE_COLUMNS) ^ set(DESCR)
feature_dictionary = pd.DataFrame({
    "feature": FEATURE_COLUMNS,
    "family": [FAMILY_OF[f] for f in FEATURE_COLUMNS],
    "description": [DESCR[f] for f in FEATURE_COLUMNS],
    "missing_value": [missing_meaning(f) for f in FEATURE_COLUMNS],
})
feature_dictionary.to_csv(P3_DIR / "feature_dictionary_part3.csv", index=False)
print("Saved", P3_DIR / "feature_dictionary_part3.csv")
display(feature_dictionary.head(12))

Saved /content/drive/MyDrive/BT4222 Final/JDSearch/processed/part3_features/feature_dictionary_part3.csv


,feature,family,description,missing_value
0,exact_wid_click_0_7d,A,Clicks on this exact product 0-7 days before t...,never missing (0 = none)
1,exact_wid_cart_0_7d,A,Cart additions of this exact product 0-7 days ...,never missing (0 = none)
2,exact_wid_ord_0_7d,A,Orders of this exact product 0-7 days before t...,never missing (0 = none)
3,exact_wid_cart_7_90d,A,Cart additions of this exact product 7-90 days...,never missing (0 = none)
4,exact_wid_ord_7_90d,A,Orders of this exact product 7-90 days before ...,never missing (0 = none)
5,exact_wid_ord_90_365d,A,Orders of this exact product 90-365 days befor...,never missing (0 = none)
6,exact_wid_last_seen_days,A,Days since the most recent action on this exac...,missing = no evidence (value never seen in the...
7,brand_aff,B,Recency- and action-weighted sum of history ac...,missing = no evidence (value never seen in the...
8,brand_share,B,brand_aff as a share of the session's total hi...,missing = no evidence (value never seen in the...
9,brand_last_seen_days,B,Days since the candidate's brand last occurred...,missing = no evidence (value never seen in the...


## Step 8 — Evaluate features (before any model is trained)

This step applies the checks in Section 6 of the proposal. None of them trains a model. Each compares a feature's values with the labels already in the data, on a random sample of 10,000 **training** sessions. The validation and test sessions are not looked at. The question for each feature is whether there is evidence that it can put engaged candidates above ignored ones **within the same slate**, because that is the only thing NDCG rewards.

The sample is loaded first. The display position of each candidate is attached from the Step 1 files for the leakage check only; it is not a feature.

In [15]:
rng = np.random.default_rng(SEED)
train_sessions = sessions.loc[sessions["split"] == "train", "session_id"].to_numpy()
sample_ids = np.sort(rng.choice(train_sessions, size=min(SCREEN_SESSIONS, len(train_sessions)), replace=False))
parts = []
for i in range(N_CHUNKS):
    f = pd.read_parquet(FEAT_DIR / f"feat_{i:03d}.parquet")
    pos = pd.read_parquet(STAGE_DIR / f"cand_{i:03d}.parquet", columns=["session_id", "position"])
    assert (pos["session_id"].to_numpy() == f["session_id"].to_numpy()).all()
    keep = isin_sorted(f["session_id"].to_numpy(), sample_ids)
    if keep.any():
        parts.append(f[keep].assign(position=pos["position"].to_numpy()[keep]))
    del f, pos
screen = pd.concat(parts, ignore_index=True).sort_values(["session_id"], kind="stable").reset_index(drop=True)
del parts; gc.collect()
screen["engaged"] = screen["label"] > 0
print(f"Evaluation sample: {screen['session_id'].nunique():,} training sessions, {len(screen):,} candidate rows, "
      f"{screen['engaged'].mean():.2%} engaged")
ram("evaluation sample loaded")

Evaluation sample: 10,000 training sessions, 890,023 candidate rows, 2.08% engaged
[RAM] evaluation sample loaded: 6.65 GB in use


### 8.1 Does the underlying behaviour exist?

**What is computed.** For each type of match, the share of *engaged* candidates (label above 0) that match the user's history is compared with the share of *ignored* candidates that do, within the same sessions. The types of match are: the exact product; the brand, the shop and each category level; any shared title term; and having an item2vec vector.

**How to read it.** A clearly higher share among engaged candidates (a ratio well above 1) supports the premise of the feature family: users tend to engage with products that resemble their past behaviour in that respect. A ratio near 1 means that the match is just as common among ignored candidates, and that family is unlikely to help whatever model is used. For the category levels, also look at the absolute shares: if almost every candidate matches, the match cannot separate candidates within a slate (see 8.2).

In [16]:
indicators = {
    "exact product in history": screen["exact_wid_last_seen_days"].notna(),
    **{f"{k} seen in history": screen[f"{k}_aff"].notna() for k in ATTRS},
    "shares a title term with a history product": screen["title_cos_max"] > 0,
    "product has an item2vec vector": screen["i2v_cos_max"].notna(),
    "brand seen in query-matched history": screen["qhist_brand_aff"].notna(),
}
premise = pd.DataFrame({
    "share of engaged candidates": {k: v[screen["engaged"]].mean() for k, v in indicators.items()},
    "share of ignored candidates": {k: v[~screen["engaged"]].mean() for k, v in indicators.items()},
})
premise["ratio (engaged / ignored)"] = premise.iloc[:, 0] / premise.iloc[:, 1]
display(premise.style.format({"share of engaged candidates": "{:.2%}", "share of ignored candidates": "{:.2%}",
                              "ratio (engaged / ignored)": "{:.2f}"}))

,share of engaged candidates,share of ignored candidates,ratio (engaged / ignored)
exact product in history,9.56%,0.58%,16.36
brand seen in history,31.40%,21.72%,1.45
shop seen in history,21.65%,6.61%,3.28
cate1 seen in history,78.03%,75.04%,1.04
cate2 seen in history,62.81%,57.21%,1.10
cate3 seen in history,47.75%,39.92%,1.20
cate4 seen in history,45.87%,37.15%,1.23
shares a title term with a history product,99.20%,99.10%,1.00
product has an item2vec vector,56.32%,25.11%,2.24
brand seen in query-matched history,15.65%,10.36%,1.51


### 8.2 Coverage and variation within the slate

**What is computed.**
- **Coverage**: the share of rows on which the feature is not missing.
- **Within-slate variation**: the share of sessions in which the feature takes at least two different values (a missing value counts as a value) across the session's candidates.

**How to read it.** A feature can change the order of a slate only if it differs between that slate's candidates. A feature that is constant within most slates, such as a session-level context variable or a category that every candidate shares, cannot reorder anything by itself, however much it varies between sessions. Context variables are expected to have zero within-slate variation; they are kept because the model can combine them with the candidate-level features.

In [17]:
coverage = screen[FEATURE_COLUMNS].notna().mean()
variation = screen.groupby("session_id")[FEATURE_COLUMNS].nunique(dropna=False).ge(2).mean()
cov_var = pd.DataFrame({"family": [FAMILY_OF[f] for f in FEATURE_COLUMNS],
                        "coverage": coverage, "varies within slate": variation})
display(cov_var.groupby("family")[["coverage", "varies within slate"]].mean().style.format("{:.1%}"))

,coverage,varies within slate
family,,
A,85.8%,10.4%
B,39.8%,57.4%
C,97.3%,95.2%
D,44.2%,93.7%
E,78.3%,61.0%
F,57.6%,16.2%
G,99.8%,94.0%
H,59.2%,69.4%
context,100.0%,0.0%


### 8.3 Ranking ability of each feature on its own

**What is computed.**
- **Within-session AUC.** For one engaged and one ignored candidate drawn at random from the same session, it is the probability that the engaged candidate has the higher value of the feature. Ties count one half, and missing values rank lowest. The AUC is computed per session and averaged over sessions. 0.5 means no information. A value below 0.5 means that *lower* values go with engagement, as expected for features measured in days, and is just as informative. The column `|AUC − 0.5|` is the strength regardless of direction.
- **AUC on rows without an exact match.** The same statistic computed only on candidates whose product is *not* in the user's history. This covers about 99% of rows, where family A has nothing to say, and is where the coarser features must show their value.
- **Single-feature NDCG@10.** Each slate is sorted by the feature alone, in its informative direction, with ties broken at random, and NDCG@10 is computed exactly as in Part 2's `evaluate_ranking` (relevance = label above 0). It is compared with a random order and with the training-set product rate `item_ctr_train`. Sorting by a feature is not a trained model; it shows directly, in the metric that will be reported, how far the feature alone would take the ranking.

The NDCG function used here is a vectorised version of Part 2's `evaluate_ranking`. The next cell checks that the two give identical results.

In [18]:
def dcg_at_k(ranks, k=10):
    return sum(1.0 / np.log2(r + 1) for r in ranks if r <= k)


def evaluate_ranking_part2(df, score_col):
    # Part 2's evaluate_ranking (merged notebook, last cell), returning the metrics instead of printing them.
    d = df[['session_id', score_col, 'label']].copy()
    np.random.seed(42)
    d['tiebreak'] = np.random.rand(len(d))
    d = d.sort_values(['session_id', score_col, 'tiebreak'], ascending=[True, False, False])
    d['rank'] = d.groupby('session_id').cumcount() + 1
    d['relevant'] = d['label'] > 0
    grouped = d[d['relevant']].groupby('session_id')['rank'].apply(list)
    ndcg, mrr, p1 = [], [], []
    for sid, ranks in grouped.items():
        a = dcg_at_k(ranks, 10); i = dcg_at_k(list(range(1, len(ranks) + 1)), 10)
        ndcg.append(a / i if i > 0 else 0); mrr.append(1.0 / min(ranks)); p1.append(1 if min(ranks) == 1 else 0)
    return np.mean(ndcg), np.mean(mrr), np.mean(p1)


def ranking_metrics(session_id, score, label, k=10, seed=42):
    # Vectorised NDCG@k, MRR and P@1 with the same definition, ordering and tie-breaking as Part 2.
    np.random.seed(seed)
    tb = np.random.rand(len(score))
    s = np.where(np.isnan(score), -np.inf, score)
    order = np.lexsort((-tb, -s, session_id))
    sid, rel = session_id[order], (label[order] > 0)
    first = np.r_[True, sid[1:] != sid[:-1]]
    grp = np.cumsum(first) - 1
    rank = np.arange(len(sid)) - np.flatnonzero(first)[grp] + 1
    gain = np.where(rel & (rank <= k), 1.0 / np.log2(rank + 1), 0.0)
    n_g = grp[-1] + 1
    dcg = np.bincount(grp, weights=gain, minlength=n_g)
    n_rel = np.bincount(grp, weights=rel, minlength=n_g)
    disc = np.r_[0, np.cumsum(1.0 / np.log2(np.arange(2, k + 2)))]
    idcg = disc[np.minimum(n_rel, k).astype(int)]
    best = np.full(n_g, np.inf); np.minimum.at(best, grp[rel], rank[rel])
    has = n_rel > 0
    return (dcg[has] / idcg[has]).mean(), (1.0 / best[has]).mean(), (best[has] == 1).mean()


chk = screen[screen["session_id"].isin(sample_ids[:500])]
a = evaluate_ranking_part2(chk, "brand_aff")
b = ranking_metrics(chk["session_id"].to_numpy(), chk["brand_aff"].to_numpy(np.float64), chk["label"].to_numpy())
print("Part 2 evaluate_ranking:", np.round(a, 6), " vectorised:", np.round(b, 6))
assert np.allclose(a, b), "the vectorised metric does not match Part 2's definition"

Part 2 evaluate_ranking: [0.120924 0.13791  0.044   ]  vectorised: [0.120924 0.13791  0.044   ]


In [19]:
def session_auc(session_id, x, engaged):
    # Mean over sessions of the within-session AUC (ties = 1/2, missing values rank lowest).
    x = np.where(np.isnan(x), -np.inf, x)
    r = pd.Series(x).groupby(session_id).rank(method="average").to_numpy()
    g = pd.DataFrame({"s": session_id, "r": r * engaged, "p": engaged, "n": ~engaged}).groupby("s")[["r", "p", "n"]].sum()
    ok = (g["p"] > 0) & (g["n"] > 0)
    g = g[ok]
    auc = (g["r"] - g["p"] * (g["p"] + 1) / 2) / (g["p"] * g["n"])
    return auc.mean()


sid = screen["session_id"].to_numpy()
lab = screen["label"].to_numpy()
eng = screen["engaged"].to_numpy()
no_exact = screen["exact_wid_last_seen_days"].isna().to_numpy()
rows = []
for f in FEATURE_COLUMNS:
    x = screen[f].to_numpy(np.float64)
    auc = session_auc(sid, x, eng)
    auc_ne = session_auc(sid[no_exact], x[no_exact], eng[no_exact])
    direction = 1.0 if (np.isnan(auc) or auc >= 0.5) else -1.0
    ndcg = ranking_metrics(sid, direction * x, lab)[0]
    rows.append({"feature": f, "family": FAMILY_OF[f], "AUC": auc, "|AUC-0.5|": abs(auc - 0.5),
                 "AUC (no exact match)": auc_ne, "direction": "higher" if direction > 0 else "lower",
                 "single-feature NDCG@10": ndcg})
ranking = pd.DataFrame(rows).set_index("feature")
base_random = ranking_metrics(sid, np.zeros(len(sid)), lab)[0]
base_ctr = ranking.loc["item_ctr_train", "single-feature NDCG@10"]
print(f"Reference: random order NDCG@10 = {base_random:.4f};  item_ctr_train alone = {base_ctr:.4f}")
display(ranking.sort_values("|AUC-0.5|", ascending=False).head(30).style.format(
    {"AUC": "{:.3f}", "|AUC-0.5|": "{:.3f}", "AUC (no exact match)": "{:.3f}", "single-feature NDCG@10": "{:.4f}"}))
print("Strongest feature of each family:")
display(ranking.sort_values("|AUC-0.5|", ascending=False).groupby("family").head(1)
        [["family", "AUC", "AUC (no exact match)", "single-feature NDCG@10"]])

Reference: random order NDCG@10 = 0.0815;  item_ctr_train alone = 0.2578


,family,AUC,|AUC-0.5|,AUC (no exact match),direction,single-feature NDCG@10
feature,,,,,,
item_impr_train,G,0.760,0.260,0.756,higher,0.3217
i2v_cos_max_slate_pct,H,0.695,0.195,0.704,higher,0.2347
i2v_cos_max,D,0.679,0.179,0.656,higher,0.2180
i2v_cos_max_slate_z,H,0.676,0.176,0.653,higher,0.2127
i2v_cos_last5,D,0.673,0.173,0.659,higher,0.1870
i2v_cos_mean,D,0.662,0.162,0.653,higher,0.1611
title_cos_max,C,0.642,0.142,0.603,higher,0.2188
title_cos_max_slate_z,H,0.642,0.142,0.603,higher,0.2188
last_i2v_cos,E,0.642,0.142,0.628,higher,0.1770


Strongest feature of each family:


,family,AUC,AUC (no exact match),single-feature NDCG@10
feature,,,,
item_impr_train,G,0.760050,0.755784,0.321659
i2v_cos_max_slate_pct,H,0.694591,0.703857,0.234665
i2v_cos_max,D,0.679055,0.655611,0.218046
title_cos_max,C,0.642309,0.602764,0.218759
last_i2v_cos,E,0.641867,0.628417,0.176957
shop_share,B,0.586285,0.546906,0.169133
exact_wid_last_seen_days,A,0.554699,0.500000,0.170476
qhist_wid_cnt,F,0.526953,0.500000,0.129932
click_count_0_7d,context,0.500000,0.500000,0.081466


### 8.3 (continued) Lift tables

**What is computed.** For a few central features, the engagement rate in each band of the feature: missing, then five quantile bands of the non-missing values. This is the most direct evidence to quote in a report. For example: "candidates whose brand the user has ordered before were engaged with X% of the time, against Y% for candidates with an unseen brand".

In [20]:
LIFT_FEATURES = ["exact_wid_last_seen_days", "brand_aff", "cate3_aff", "title_cos_max", "i2v_cos_max", "item_ctr_train"]
lift_tables = []
for f in LIFT_FEATURES:
    x = screen[f]
    band = pd.Series("missing", index=screen.index, dtype=object)
    nn = x.notna()
    if nn.sum() > 0:
        band[nn] = pd.qcut(x[nn].rank(method="first"), q=min(5, nn.sum()), labels=False).map(lambda q: f"Q{q + 1}")
    t = screen.groupby(band)["engaged"].agg(["mean", "size"])
    t["share of rows"] = t["size"] / len(screen)
    t["value range"] = [("-" if b == "missing" else f"{x[band == b].min():.3g} to {x[band == b].max():.3g}") for b in t.index]
    t = t.rename(columns={"mean": "engagement rate"}).drop(columns="size")
    t.index = pd.MultiIndex.from_product([[f], t.index], names=["feature", "band"])
    lift_tables.append(t)
lift = pd.concat(lift_tables)
lift.to_csv(P3_DIR / "feature_lift_tables.csv")
display(lift.style.format({"engagement rate": "{:.2%}", "share of rows": "{:.1%}"}))

### 8.4 Additional information: redundancy

**What is computed.** Spearman rank correlations between all pairs of features, on a random 100,000 rows of the sample. Pairs with an absolute correlation above 0.95 are listed.

**How to read it.** Two features that are almost perfectly correlated carry essentially the same information, so one of them adds nothing. A flag here is a prompt to keep the simpler or more interpretable feature, not an automatic removal; LightGBM is not harmed by mild redundancy. The AUC on rows without an exact match (8.3) is the other half of this check: it shows whether a feature carries information where family A is silent.

In [21]:
sub = screen[FEATURE_COLUMNS].sample(n=min(100_000, len(screen)), random_state=SEED)
with np.errstate(invalid="ignore", divide="ignore"):
    corr = sub.rank().corr()        # Spearman = Pearson correlation of ranks (pairwise complete)
del sub
pairs = corr.where(np.triu(np.ones(corr.shape, bool), k=1)).stack()
redundant = pairs[pairs.abs() > 0.95].sort_values(key=np.abs, ascending=False)
print(f"{len(redundant)} feature pairs with |Spearman| > 0.95")
display(redundant.rename("Spearman").to_frame().head(40))
redundant_with = {f: [] for f in FEATURE_COLUMNS}
for (a, b), r in redundant.items():
    redundant_with[a].append(b); redundant_with[b].append(a)

7 feature pairs with |Spearman| > 0.95


,,Spearman
cate3_last_seen_days,cate4_last_seen_days,0.985308
title_cos_profile_slate_pct,title_cos_profile_slate_z,-0.984692
cate3_share,cate4_share,0.980940
title_cos_max_slate_pct,title_cos_max_slate_z,-0.980631
i2v_cos_max_slate_pct,i2v_cos_max_slate_z,-0.977090
cate3_aff,cate4_aff,0.971250
cate3_ord_cnt,cate4_ord_cnt,0.970844


### 8.5 Leakage

**What is computed.** The correlation of each feature with the candidate's display position, measured **within slates**. Both the feature and the position are first converted to percentile ranks inside each session, and the two ranks are then correlated across all rows. Measuring within slates matters: a raw correlation would be dominated by differences between sessions, for example larger slates simply contain larger positions, and would flag `slate_size` for no good reason. The first displayed candidate is positive in about 99% of sessions, so position is a near-perfect predictor that must not reach the model. A feature that follows the display order within slates may be acting as a proxy for it. The flag threshold is 0.3, in absolute value. Session-level features cannot follow the order within a slate, and receive 0.

**How to read it.** A moderate correlation is not necessarily leakage. The search engine ranks products partly by popularity and relevance, so a legitimate popularity prior such as `item_ctr_train` can correlate with position. Flags here call for a review of how the feature is built, not automatic removal. Separately, any feature whose AUC departs from 0.5 by more than 0.4 is flagged as *suspicious*: no single legitimate similarity feature is expected to rank that well, and such a value usually means a construction error that lets the label in.

In [22]:
gs = screen.groupby("session_id")
pos_rank = gs["position"].rank(pct=True)
with np.errstate(invalid="ignore", divide="ignore"):
    feat_rank = gs[FEATURE_COLUMNS].rank(pct=True)
    pos_corr = feat_rank.corrwith(pos_rank).fillna(0.0)     # constant within slates -> no correlation
del feat_rank
print("Largest within-slate correlations with display position:")
display(pos_corr.abs().sort_values(ascending=False).head(10).rename("|within-slate corr. with position|").to_frame())

Largest within-slate correlations with display position:


,|within-slate corr. with position|
item_impr_train,0.057630
exact_wid_last_seen_days,0.046455
exact_wid_click_0_7d,0.034589
shop_share,0.032146
shop_aff,0.032146
qhist_wid_cnt,0.028054
shop_aff_slate_z,0.027549
title_cos_max_slate_pct,0.027142
title_cos_max_slate_z,0.026713
title_cos_max,0.026686


### 8.6 Summary table, flags and the recommended feature list

All the statistics above are combined into one row per feature and saved as `feature_screening.csv`. Each feature receives these flags:

| Flag | Rule | Meaning |
|---|---|---|
| `weak` | \|AUC − 0.5\| < 0.02 | Little ranking information on its own (it may still help in combination) |
| `low_variation` | varies within fewer than 20% of slates | Rarely able to reorder a slate (expected for context variables) |
| `redundant` | \|Spearman\| > 0.95 with another feature | Duplicates another feature |
| `position_proxy` | \|within-slate correlation with position\| > 0.3 | Review for leakage |
| `suspicious` | \|AUC − 0.5\| > 0.4 | Too strong to be plausible: check for a construction error |

By default, all features are kept (`DROP_FLAGGED = False`). The flags are reported so that the choice can be made deliberately. If `DROP_FLAGGED` is set to `True`, features flagged `suspicious` or `position_proxy` are excluded, together with any listed in `MANUAL_DROP`. The resulting list is saved as `model_features.json`, which the training example in Step 10 reads.

In [23]:
DROP_FLAGGED = False
MANUAL_DROP = []          # e.g. ["title_hist_qry_overlap"] if query and title terms turn out not to share IDs

summary = cov_var.join(ranking.drop(columns="family")).join(pos_corr.rename("within-slate corr. with position"))
summary["flag_weak"] = summary["|AUC-0.5|"] < 0.02
summary["flag_low_variation"] = summary["varies within slate"] < 0.20
summary["flag_redundant"] = [len(redundant_with[f]) > 0 for f in summary.index]
summary["redundant_with"] = [", ".join(redundant_with[f]) for f in summary.index]
summary["flag_position_proxy"] = summary["within-slate corr. with position"].abs() > 0.3
summary["flag_suspicious"] = summary["|AUC-0.5|"] > 0.4
summary.index.name = "feature"
summary.to_csv(P3_DIR / "feature_screening.csv")

drop = set(MANUAL_DROP)
if DROP_FLAGGED:
    drop |= set(summary.index[summary["flag_suspicious"] | summary["flag_position_proxy"]])
model_features = [f for f in FEATURE_COLUMNS if f not in drop]
(P3_DIR / "model_features.json").write_text(json.dumps(model_features, indent=1))

flag_cols = [c for c in summary.columns if c.startswith("flag_")]
print("Number of features per flag:"); print(summary[flag_cols].sum().to_string())
print(f"\nFeatures recommended for training: {len(model_features)} of {len(FEATURE_COLUMNS)}  (excluded: {sorted(drop)})")
fam_table = summary.groupby("family").agg(features=("AUC", "size"), mean_coverage=("coverage", "mean"),
                                          best_abs_auc=("|AUC-0.5|", "max"),
                                          best_single_ndcg=("single-feature NDCG@10", "max"))
display(fam_table.style.format({"mean_coverage": "{:.1%}", "best_abs_auc": "{:.3f}", "best_single_ndcg": "{:.4f}"}))
print("Query-term check: share of candidates with any title term in the user's past queries =",
      f"{(screen['title_hist_qry_overlap'] > 0).mean():.2%}  (near 0 would suggest that query and title terms use different IDs)")

Number of features per flag:
flag_weak              34
flag_low_variation     31
flag_redundant         14
flag_position_proxy     0
flag_suspicious         0

Features recommended for training: 90 of 90  (excluded: [])


,features,mean_coverage,best_abs_auc,best_single_ndcg
family,,,,
A,7,85.8%,0.055,0.1705
B,24,39.8%,0.086,0.1691
C,4,97.3%,0.142,0.2188
D,4,44.2%,0.179,0.2180
E,4,78.3%,0.142,0.1770
F,4,57.6%,0.027,0.1299
G,4,99.8%,0.260,0.3217
H,16,59.2%,0.195,0.2578
context,23,100.0%,0.000,0.0815


Query-term check: share of candidates with any title term in the user's past queries = 71.20%  (near 0 would suggest that query and title terms use different IDs)


## Step 9 — Consistency checks against the earlier notebooks' outputs

This notebook does not read the earlier outputs to build anything. Where they exist, they are compared with what this notebook rebuilt, to confirm that the definitions agree:

1. **Split.** The `split` assigned in Step 2 is compared session by session with `processed/user_split.csv`.
2. **Session features (context).** The 15 Part 1 columns are compared with `processed/session_recency_features.parquet`. The maximum absolute difference should be 0 for counts and negligible for days. Days are stored here as 32-bit floats, so differences of about 10⁻⁴ days are rounding.
3. **Exact-product features (family A).** Column totals and the number of matched rows are compared with `processed/candidate_recency_chunks/`. Row order differs between the two because the candidates were shuffled with different random draws, so totals are compared rather than individual rows.

A missing file is reported and skipped.

In [24]:
split_file = OUTPUT_DIR / "user_split.csv"
if split_file.exists():
    old = pd.read_csv(split_file).set_index("session_id")["split"]
    new = sessions.set_index("session_id")["split"]
    common = old.index.intersection(new.index)
    print(f"Split: {len(common):,} common sessions; identical assignment: {(old[common] == new[common]).all()};"
          f" sessions only in one of them: {len(old.index.symmetric_difference(new.index))}")
else:
    print("user_split.csv not found - split check skipped")

recency_file = OUTPUT_DIR / "session_recency_features.parquet"
part1_cols = FAMILIES["context"][:15]
if recency_file.exists():
    old = pd.read_parquet(recency_file).set_index("session_id")
    new = pd.concat([pd.read_parquet(FEAT_DIR / f"feat_{i:03d}.parquet", columns=["session_id"] + part1_cols)
                     .drop_duplicates("session_id") for i in range(N_CHUNKS)]).set_index("session_id")
    common = old.index.intersection(new.index)
    diffs = {c: float(np.nanmax(np.abs(old.loc[common, c].to_numpy(np.float64) - new.loc[common, c].to_numpy(np.float64))))
             for c in part1_cols if c in old.columns}
    print(f"Session features: {len(common):,} common sessions; largest absolute difference per column:")
    print(pd.Series(diffs).to_string())
else:
    print("session_recency_features.parquet not found - check skipped")

exact_dir = OUTPUT_DIR / "candidate_recency_chunks"
a_cols = FAMILIES["A"]
if exact_dir.exists() and any(exact_dir.glob("*.parquet")):
    old_tot = pd.Series(0.0, index=a_cols[:-1]); old_match = 0
    for f in sorted(exact_dir.glob("*.parquet")):
        d = pd.read_parquet(f, columns=a_cols)
        old_tot += d[a_cols[:-1]].sum(); old_match += d["exact_wid_last_seen_days"].notna().sum()
    new_tot = pd.Series(0.0, index=a_cols[:-1]); new_match = 0
    for i in range(N_CHUNKS):
        d = pd.read_parquet(FEAT_DIR / f"feat_{i:03d}.parquet", columns=a_cols)
        new_tot += d[a_cols[:-1]].sum(); new_match += d["exact_wid_last_seen_days"].notna().sum()
    print(pd.DataFrame({"Part 1": old_tot, "this notebook": new_tot}).to_string())
    print(f"Rows with an exact match: Part 1 {old_match:,}; this notebook {new_match:,}")
else:
    print("candidate_recency_chunks not found - check skipped")

Split: 171,175 common sessions; identical assignment: True; sessions only in one of them: 0
Session features: 171,175 common sessions; largest absolute difference per column:
click_count_0_7d               0.000000
cart_count_0_7d                0.000000
ord_count_0_7d                 0.000000
flw_count_0_7d                 0.000000
click_count_7_90d              0.000000
cart_count_7_90d               0.000000
ord_count_7_90d                0.000000
flw_count_7_90d                0.000000
click_count_90_365d            0.000000
cart_count_90_365d             0.000000
ord_count_90_365d              0.000000
flw_count_90_365d              0.000000
history_len_clean              0.000000
history_span_days              0.000015
days_since_last_interaction    0.000015
                         Part 1  this notebook
exact_wid_click_0_7d   208092.0       208092.0
exact_wid_cart_0_7d     29669.0        29669.0
exact_wid_ord_0_7d       1809.0         1809.0
exact_wid_cart_7_90d    36412.0      

## Step 10 — How to train LightGBM with these outputs (example, not run)

The features are now on Drive under `processed/part3_features/`:

| File | Contents |
|---|---|
| `feature_chunks/feat_000.parquet` … `feat_085.parquet` | One row per candidate: `session_id`, `wid`, `label`, `split` and the 90 features (float32). The rows of a session are contiguous. |
| `feature_dictionary_part3.csv` | Family, definition and meaning of a missing value, for each feature |
| `feature_screening.csv`, `feature_lift_tables.csv` | The results of Step 8 |
| `model_features.json` | The feature list recommended by Step 8 |

The ranker described in Section 7 of the proposal would be trained as follows. This is an example only; it is not run in this notebook.

**1. Load one split at a time, with only the needed columns.** Rows must be sorted by session, so that each session's candidates are contiguous. The group sizes tell LightGBM where each slate starts and ends.

```python
import json, numpy as np, pandas as pd, lightgbm as lgb

FEATS = json.loads((P3_DIR / "model_features.json").read_text())

def load_split(split, session_fraction=1.0, seed=42):
    parts = []
    for f in sorted(FEAT_DIR.glob("feat_*.parquet")):
        d = pd.read_parquet(f, columns=["session_id", "label", "split"] + FEATS)
        parts.append(d[d["split"] == split].drop(columns="split"))
    d = pd.concat(parts, ignore_index=True).sort_values("session_id", kind="stable")
    if session_fraction < 1.0:   # keep whole sessions only, to save memory
        keep = d["session_id"].drop_duplicates().sample(frac=session_fraction, random_state=seed)
        d = d[d["session_id"].isin(keep)]
    groups = d.groupby("session_id", sort=False).size().to_numpy()
    return d[FEATS].to_numpy(np.float32), d["label"].to_numpy(), groups, d["session_id"].to_numpy()

X_tr, y_tr, g_tr, _ = load_split("train")        # about 12.3M rows x 90 features = 4.4 GB as float32
dtrain = lgb.Dataset(X_tr, y_tr, group=g_tr, feature_name=FEATS, free_raw_data=True)
dtrain.construct(); del X_tr                      # LightGBM keeps a compact binned copy (about 1 GB)
X_va, y_va, g_va, _ = load_split("val")
dval = lgb.Dataset(X_va, y_va, group=g_va, reference=dtrain)
```

If memory is short, pass `session_fraction=0.5` to `load_split("train", ...)`. Whole sessions are always kept together.

**2. Train with LambdaRank and stop early on validation NDCG@10.** The graded labels 0–3 are used as relevance levels with gains 0, 1, 3 and 7, so that ranking an order above a click is rewarded. Missing values are left as `NaN`, because LightGBM learns a separate direction for them.

```python
params = dict(objective="lambdarank", metric="ndcg", eval_at=[10], label_gain=[0, 1, 3, 7],
              learning_rate=0.05, num_leaves=63, min_data_in_leaf=200,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, seed=42, verbose=-1)
ranker = lgb.train(params, dtrain, num_boost_round=2000, valid_sets=[dval], valid_names=["val"],
                   callbacks=[lgb.early_stopping(100), lgb.log_evaluation(50)])
ranker.save_model(str(P3_DIR / "lgbm_ranker.txt"))
```

**3. Evaluate once on the test split, with the same metric as Part 2,** and compare with the leakage-free popularity baseline:

```python
X_te, y_te, g_te, sid_te = load_split("test")
score = ranker.predict(X_te, num_iteration=ranker.best_iteration)
print("LightGBM:", ranking_metrics(sid_te, score, y_te))                      # NDCG@10, MRR, P@1
print("Popularity (train-only):", ranking_metrics(sid_te, X_te[:, FEATS.index("item_ctr_train")], y_te))
```

**4. Interpret.** Report the gain importance (`ranker.feature_importance("gain")`) by family. Report test NDCG@10 separately for sessions with an exact match, with only a brand match, and with neither. The family ablations described in Section 7 of the proposal retrain the model without one family at a time and compare validation NDCG@10 with a paired bootstrap over sessions.